In [7]:
%matplotlib notebook
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

plt.rcParams['figure.max_open_warning'] = 0  # suppress warning for interactive backend

In [ ]:
# ── Rotation axis (lab frame) ─────────────────────────────────────────────────
Theta_axis = 7.4   # degrees — polar angle of rotation axis
Phi_axis   = 0.0   # degrees — azimuthal angle of rotation axis

# ── Cone half-angle ───────────────────────────────────────────────────────────
Lambda     = 30.0   # degrees — angle between rod and rotation axis

# ── Optical parameters ────────────────────────────────────────────────────────
NA = 1.3            # numerical aperture (uncalibrated)
nw = 1.33           # refractive index of immersion medium

# ── Resolution of the rotation phase sweep ───────────────────────────────────
N_psi = 360


In [ ]:
def fourkas_ABC(alpha):
    """Coefficients A, B, C as a function of collection half-angle α."""
    ca = np.cos(alpha)
    A  = 1/6  - ca/4       + ca**3/12
    B  =        ca/8       - ca**3/8
    C  = 7/48 - ca/16 - ca**2/16 - ca**3/48
    return A, B, C


def theta_to_r(theta_rad, A, B, C):
    """Forward Fourkas: polar angle θ → anisotropy radius r.

    From the Fourkas (2001) formula:
        ax = C·sin²θ·cos2φ / (A + B·sin²θ)
        ay = C·sin²θ·sin2φ / (A + B·sin²θ)
        r  = C·sin²θ        / (A + B·sin²θ)
    """
    s2 = np.sin(theta_rad) ** 2
    return C * s2 / (A + B * s2)


def r_to_theta(r, A, B, C):
    """Inverse Fourkas: anisotropy radius r → polar angle θ.

    Invert  r = C·sin²θ / (A + B·sin²θ)  →  sin²θ = r·A / (C − r·B)
    Clip r at the saturation value R_SAT = C/(A + B) reached as θ→90°.
    """
    R_SAT = C / (A + B)
    r_clip = np.clip(r, 0.0, R_SAT)
    s2 = r_clip * A / (C - r_clip * B)
    s2 = np.clip(s2, 0.0, 1.0)
    return np.arcsin(np.sqrt(s2))


# ── Compute and display coefficients ─────────────────────────────────────────
alpha = np.arcsin(NA / nw)          # collection half-angle (radians)
A, B, C = fourkas_ABC(alpha)
R_SAT = C / (A + B)

print(f"Collection half-angle α = {np.degrees(alpha):.4f}°")
print(f"Fourkas A = {A:.6f}")
print(f"Fourkas B = {B:.6f}")
print(f"Fourkas C = {C:.6f}")
print(f"Saturation anisotropy R_SAT = C/(A+B) = {R_SAT:.6f}")


In [ ]:
# ── Convert parameters to radians ────────────────────────────────────────────
theta_axis = np.radians(Theta_axis)
phi_axis   = np.radians(Phi_axis)
lam        = np.radians(Lambda)

# ── Rotation axis unit vector k̂ ──────────────────────────────────────────────
kx = np.sin(theta_axis) * np.cos(phi_axis)
ky = np.sin(theta_axis) * np.sin(phi_axis)
kz = np.cos(theta_axis)
k_hat = np.array([kx, ky, kz])

# ── Initial rod direction (at ψ=0): a vector at angle Λ from k̂) ─────────────
# Build an orthonormal basis (k̂, e1, e2) so e1 is the initial perpendicular.
# Choose e1 in the plane of k̂ and ẑ (or fall back to x̂ if k̂‖ẑ).
if abs(kz) < 0.9999:
    z_hat = np.array([0.0, 0.0, 1.0])
    e1 = z_hat - kz * k_hat           # component of ẑ perpendicular to k̂
    e1 = e1 / np.linalg.norm(e1)
else:
    e1 = np.array([1.0, 0.0, 0.0])   # k̂ ≈ ẑ — use x̂

e2 = np.cross(k_hat, e1)              # complete right-hand basis

# ── Rodrigues rotation ────────────────────────────────────────────────────────
# Rod unit vector for phase ψ:
#   d(ψ) = cos(Λ)·k̂ + sin(Λ)·[cos(ψ)·e1 + sin(ψ)·e2]
psi_arr = np.linspace(0.0, 2.0 * np.pi, N_psi, endpoint=False)

d = (  np.cos(lam) * k_hat[np.newaxis, :]
     + np.sin(lam) * (np.cos(psi_arr)[:, np.newaxis] * e1[np.newaxis, :]
                    + np.sin(psi_arr)[:, np.newaxis] * e2[np.newaxis, :]) )
# d has shape (N_psi, 3); each row is a unit vector

dx, dy, dz = d[:, 0], d[:, 1], d[:, 2]

# ── Lab-frame polar / azimuthal angles ───────────────────────────────────────
theta_psi = np.arccos(np.clip(dz, -1.0, 1.0))   # θ(ψ) ∈ [0, π]
phi_psi   = np.arctan2(dy, dx)                   # φ(ψ) ∈ (−π, π]

# ── Forward Fourkas: θ → r → (ax, ay) ────────────────────────────────────────
r_true = theta_to_r(theta_psi, A, B, C)
ax_true = r_true * np.cos(2.0 * phi_psi)
ay_true = r_true * np.sin(2.0 * phi_psi)

# ── Inverse recovery: (ax, ay) → r_rec → θ_rec, φ_rec ───────────────────────
r_rec       = np.sqrt(ax_true**2 + ay_true**2)
theta_rec   = r_to_theta(r_rec, A, B, C)
phi_rec     = 0.5 * np.arctan2(ay_true, ax_true)

# ── Round-trip error ──────────────────────────────────────────────────────────
theta_err = np.max(np.abs(np.degrees(theta_rec - theta_psi)))

# Wrap φ difference to (−π/2, π/2] since φ and φ+π give the same orientation
dphi = phi_rec - phi_psi
dphi = (dphi + np.pi/2) % np.pi - np.pi/2
phi_err = np.max(np.abs(np.degrees(dphi)))

print(f"Max round-trip error  Δθ = {theta_err:.2e}°")
print(f"Max round-trip error  Δφ = {phi_err:.2e}°")


In [ ]:
fig_sphere = plt.figure(num='sphere', figsize=(7, 6))
fig_sphere.clf()
ax3 = fig_sphere.add_subplot(111, projection='3d')

# ── Scatter of rod endpoints coloured by ψ ───────────────────────────────────
sc = ax3.scatter(dx, dy, dz, c=psi_arr, cmap='hsv', s=8, lw=0)
plt.colorbar(sc, ax=ax3, label='ψ (rad)', pad=0.1, shrink=0.7)

# ── Rotation axis arrow ───────────────────────────────────────────────────────
ax3.quiver(0, 0, 0, kx, ky, kz, length=1.1, color='k',
           arrow_length_ratio=0.12, linewidth=2, label='rotation axis')

# ── Best-fit circle via PCA of the 3D points ─────────────────────────────────
centroid = d.mean(axis=0)
_, _, Vt = np.linalg.svd(d - centroid)
normal   = Vt[-1]                  # last singular vector = circle normal
u_vec    = Vt[0]                   # first two span the circle plane
v_vec    = Vt[1]
radius_pca = np.linalg.norm(d[0] - centroid)   # approximate radius
t_fit    = np.linspace(0, 2 * np.pi, 300)
circle_fit = (centroid[np.newaxis, :]
              + radius_pca * np.cos(t_fit)[:, np.newaxis] * u_vec[np.newaxis, :]
              + radius_pca * np.sin(t_fit)[:, np.newaxis] * v_vec[np.newaxis, :])
ax3.plot(circle_fit[:, 0], circle_fit[:, 1], circle_fit[:, 2],
         'r--', lw=1.5, label='best-fit circle (PCA)')

# ── Markers at ψ = 0°, 90°, 180°, 270° ──────────────────────────────────────
mark_labels = ['0°', '90°', '180°', '270°']
mark_indices = [0, N_psi // 4, N_psi // 2, 3 * N_psi // 4]
mark_markers = ['o', 's', 'D', '^']
for idx, mkr, lbl in zip(mark_indices, mark_markers, mark_labels):
    ax3.plot([dx[idx]], [dy[idx]], [dz[idx]], mkr, color='k', ms=10,
             mew=1.5, mfc='yellow', zorder=6, label=f'ψ = {lbl}')

ax3.set_xlim(-1, 1); ax3.set_ylim(-1, 1); ax3.set_zlim(-1, 1)
ax3.set_xlabel('x');  ax3.set_ylabel('y');  ax3.set_zlabel('z')
ax3.set_title(f'Unit sphere  Θ_axis={Theta_axis}°  Φ_axis={Phi_axis}°  Λ={Lambda}°')
ax3.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
fig_aniso = plt.figure(num='anisotropy', figsize=(6, 6))
fig_aniso.clf()
ax = fig_aniso.add_subplot(111)

sc = ax.scatter(ax_true, ay_true, c=psi_arr, cmap='hsv', s=12, lw=0, zorder=3)
plt.colorbar(sc, ax=ax, label='ψ (rad)', shrink=0.8)

# close the curve
ax.plot(np.append(ax_true, ax_true[0]), np.append(ay_true, ay_true[0]),
        'k-', lw=0.6, alpha=0.4, zorder=2)

# crosshairs at origin
ax.axhline(0, color='grey', lw=0.8, ls='--')
ax.axvline(0, color='grey', lw=0.8, ls='--')

# ── Markers at ψ = 0°, 90°, 180°, 270° ──────────────────────────────────────
mark_labels = ['0°', '90°', '180°', '270°']
mark_indices = [0, N_psi // 4, N_psi // 2, 3 * N_psi // 4]
mark_markers = ['o', 's', 'D', '^']
for idx, mkr, lbl in zip(mark_indices, mark_markers, mark_labels):
    ax.plot(ax_true[idx], ay_true[idx], mkr, color='k', ms=10,
            mew=1.5, mfc='yellow', zorder=6, label=f'ψ = {lbl}')

ax.set_xlabel(r'$a_x$');  ax.set_ylabel(r'$a_y$')
ax.set_xlim(-1, 1); ax.set_ylim(-1, 1)
ax.set_title('Anisotropy cloud  $(a_x, a_y)$')
ax.set_aspect('equal')
ax.legend(fontsize=9)
ax.grid(True, lw=0.5, alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:

# ── Compare two parameter sets ────────────────────────────────────────────────
# Set A (current parameters from above)
params_A = dict(Theta_axis=50.0, Phi_axis=0.0, Lambda=30.0, label='Set A')

# Set B — edit these to compare
params_B = dict(Theta_axis=50.0, Phi_axis=0.0, Lambda=40.0, label='Set B')

def _compute_anisotropy(Theta_axis_deg, Phi_axis_deg, Lambda_deg, N=360):
    """Compute (ax, ay) anisotropy curve for given rotation geometry."""
    th_ax = np.radians(Theta_axis_deg)
    ph_ax = np.radians(Phi_axis_deg)
    la    = np.radians(Lambda_deg)
    _kx = np.sin(th_ax) * np.cos(ph_ax)
    _ky = np.sin(th_ax) * np.sin(ph_ax)
    _kz = np.cos(th_ax)
    _k  = np.array([_kx, _ky, _kz])
    if abs(_kz) < 0.9999:
        _e1 = np.array([0.0, 0.0, 1.0]) - _kz * _k
        _e1 /= np.linalg.norm(_e1)
    else:
        _e1 = np.array([1.0, 0.0, 0.0])
    _e2 = np.cross(_k, _e1)
    _psi = np.linspace(0, 2*np.pi, N, endpoint=False)
    _d = (np.cos(la) * _k[None, :]
          + np.sin(la) * (np.cos(_psi)[:, None] * _e1[None, :]
                        + np.sin(_psi)[:, None] * _e2[None, :]))
    _theta = np.arccos(np.clip(_d[:, 2], -1, 1))
    _phi   = np.arctan2(_d[:, 1], _d[:, 0])
    _r = theta_to_r(_theta, A, B, C)
    return _r * np.cos(2*_phi), _r * np.sin(2*_phi), _psi

ax_A, ay_A, psi_A = _compute_anisotropy(params_A['Theta_axis'], params_A['Phi_axis'], params_A['Lambda'])
ax_B, ay_B, psi_B = _compute_anisotropy(params_B['Theta_axis'], params_B['Phi_axis'], params_B['Lambda'])

fig_cmp = plt.figure(num='compare', figsize=(7, 7))
fig_cmp.clf()
ax_c = fig_cmp.add_subplot(111)

# Set A
ax_c.plot(np.append(ax_A, ax_A[0]), np.append(ay_A, ay_A[0]),
          '-', lw=2, color='tab:blue',
          label=f"{params_A['label']}: Θ={params_A['Theta_axis']}° Φ={params_A['Phi_axis']}° Λ={params_A['Lambda']}°")

# Set B
ax_c.plot(np.append(ax_B, ax_B[0]), np.append(ay_B, ay_B[0]),
          '--', lw=2, color='tab:red',
          label=f"{params_B['label']}: Θ={params_B['Theta_axis']}° Φ={params_B['Phi_axis']}° Λ={params_B['Lambda']}°")

ax_c.axhline(0, color='grey', lw=0.8, ls='--')
ax_c.axvline(0, color='grey', lw=0.8, ls='--')
ax_c.set_xlabel(r'$a_x$'); ax_c.set_ylabel(r'$a_y$')
ax_c.set_xlim(-1, 1); ax_c.set_ylim(-1, 1)
ax_c.set_aspect('equal')
ax_c.set_title('Anisotropy comparison')
ax_c.legend(fontsize=9)
ax_c.grid(True, lw=0.5, alpha=0.5)
plt.tight_layout()
plt.show()


In [ ]:
psi_deg = np.degrees(psi_arr)

fig_angles = plt.figure(num='angles', figsize=(9, 6))
fig_angles.clf()
ax_t = fig_angles.add_subplot(2, 1, 1)
ax_p = fig_angles.add_subplot(2, 1, 2, sharex=ax_t)

# ── θ vs ψ ────────────────────────────────────────────────────────────────────
ax_t.plot(psi_deg, np.degrees(theta_psi),  'b-',  lw=2,   label='θ original')
ax_t.plot(psi_deg, np.degrees(theta_rec),  'r--', lw=1.5, label='θ recovered')
ax_t.set_ylabel('θ (degrees)')
ax_t.set_title('Polar angle θ and azimuthal angle φ vs rotation phase ψ')
ax_t.legend()
ax_t.grid(True, lw=0.5, alpha=0.5)

# ── φ vs ψ (unwrapped) ────────────────────────────────────────────────────────
phi_orig_uw = np.unwrap(phi_psi, period=np.pi)
phi_rec_uw  = np.unwrap(phi_rec, period=np.pi)

# align the recovered curve to the original (mod π ambiguity)
offset = np.round((phi_orig_uw[0] - phi_rec_uw[0]) / np.pi) * np.pi
phi_rec_uw += offset

ax_p.plot(psi_deg, phi_orig_uw, 'b-',  lw=2,   label='φ original (unwrapped)')
ax_p.plot(psi_deg, phi_rec_uw,  'r--', lw=1.5, label='φ recovered (unwrapped)')
ax_p.set_xlabel('ψ (degrees)')
ax_p.set_ylabel('φ (radians)')
ax_p.legend()
ax_p.grid(True, lw=0.5, alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
fig_tvp = plt.figure(num='theta_vs_phi', figsize=(7, 6))
fig_tvp.clf()
ax_tvp = fig_tvp.add_subplot(111)

ax_tvp.plot(np.degrees(phi_orig_uw), np.degrees(theta_psi), 'b-', lw=2, label='original')
ax_tvp.plot(np.degrees(phi_rec_uw), np.degrees(theta_rec), 'r--', lw=1.5, label='recovered')

# ── Markers at ψ = 0°, 90°, 180°, 270° ──────────────────────────────────────
mark_labels = ['0°', '90°', '180°', '270°']
mark_indices = [0, N_psi // 4, N_psi // 2, 3 * N_psi // 4]
mark_markers = ['o', 's', 'D', '^']
for idx, mkr, lbl in zip(mark_indices, mark_markers, mark_labels):
    ax_tvp.plot(np.degrees(phi_orig_uw[idx]), np.degrees(theta_psi[idx]), mkr,
                color='k', ms=10, mew=1.5, mfc='yellow', zorder=6, label=f'ψ = {lbl}')

ax_tvp.set_xlabel('φ (degrees)')
ax_tvp.set_ylabel('θ (degrees)')
ax_tvp.set_title('Polar angle θ vs azimuthal angle φ')
ax_tvp.legend(fontsize=9)
ax_tvp.grid(True, lw=0.5, alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
run_animation = True   # set to False to skip

if run_animation:
    # Stop previous animation if running
    _prev_anim = globals().get('anim')
    if _prev_anim and hasattr(_prev_anim, 'event_source') and _prev_anim.event_source:
        _prev_anim.event_source.stop()

    fig_anim = plt.figure(num='anim', figsize=(12, 5))
    fig_anim.clf()
    ax_sph = fig_anim.add_subplot(1, 2, 1, projection='3d')
    ax_ani = fig_anim.add_subplot(1, 2, 2)

    # ── Static background: unit sphere trajectory ─────────────────────────────
    ax_sph.scatter(dx, dy, dz, c=psi_arr, cmap='hsv', s=4, alpha=0.25, lw=0)
    ax_sph.quiver(0, 0, 0, kx, ky, kz, length=1.1, color='k',
                  arrow_length_ratio=0.12, linewidth=1.5)
    ax_sph.set_xlim(-1, 1); ax_sph.set_ylim(-1, 1); ax_sph.set_zlim(-1, 1)
    ax_sph.set_xlabel('x'); ax_sph.set_ylabel('y'); ax_sph.set_zlabel('z')
    ax_sph.set_title('Unit sphere')

    # ── Static background: anisotropy trail ──────────────────────────────────
    ax_ani.scatter(ax_true, ay_true, c=psi_arr, cmap='hsv', s=4, alpha=0.25, lw=0)
    ax_ani.axhline(0, color='grey', lw=0.8, ls='--')
    ax_ani.axvline(0, color='grey', lw=0.8, ls='--')
    ax_ani.set_xlim(-1, 1); ax_ani.set_ylim(-1, 1)
    ax_ani.set_xlabel(r'$a_x$'); ax_ani.set_ylabel(r'$a_y$')
    ax_ani.set_aspect('equal')
    ax_ani.set_title('Anisotropy $(a_x, a_y)$')
    ax_ani.grid(True, lw=0.5, alpha=0.5)

    # ── Moving dot + trail ────────────────────────────────────────────────────
    trail_len = 40

    dot_sph,   = ax_sph.plot([], [], [], 'ro', ms=8, zorder=5)
    trail_sph, = ax_sph.plot([], [], [], 'r.', ms=3, alpha=0.5)

    dot_ani,   = ax_ani.plot([], [], 'ro', ms=8, zorder=5)
    trail_ani, = ax_ani.plot([], [], 'r.', ms=3, alpha=0.5)

    def _init():
        for artist in (dot_ani, trail_ani):
            artist.set_data([], [])
        dot_sph.set_data([], [])
        dot_sph.set_3d_properties([])
        trail_sph.set_data([], [])
        trail_sph.set_3d_properties([])
        return dot_sph, trail_sph, dot_ani, trail_ani

    def _update(frame):
        lo = max(0, frame - trail_len)

        # sphere
        dot_sph.set_data([dx[frame]], [dy[frame]])
        dot_sph.set_3d_properties([dz[frame]])
        trail_sph.set_data(dx[lo:frame+1], dy[lo:frame+1])
        trail_sph.set_3d_properties(dz[lo:frame+1])

        # anisotropy
        dot_ani.set_data([ax_true[frame]], [ay_true[frame]])
        trail_ani.set_data(ax_true[lo:frame+1], ay_true[lo:frame+1])

        return dot_sph, trail_sph, dot_ani, trail_ani

    anim = FuncAnimation(
        fig_anim, _update, init_func=_init,
        frames=N_psi, interval=30, blit=False
    )
    plt.show()

## Demo for supervisor — what gets measured when a unit vector rotates around a fixed axis

**Setup.** Fix a rotation axis $\hat{\mathbf k}$ in the lab (2 angles, $\Theta_\text{axis}, \Phi_\text{axis}$). Pick an arbitrary rod orientation $\hat{\mathbf m}_0$ — a unit vector on the 2-sphere, specified directly by its **lab-frame** spherical angles $(\theta_0, \phi_0)$. The rod is rigidly attached to the rotor, so as motor phase $\psi$ advances, the orientation evolves as

$$\hat{\mathbf m}(\psi) = R_{\hat{\mathbf k}}(\psi)\,\hat{\mathbf m}_0$$

(Rodrigues rotation around $\hat{\mathbf k}$). The Fourkas signal sees only $\hat{\mathbf m}(\psi)$ — its lab-frame polar angle $\theta_\text{lab}(\psi)$ and azimuth $\phi_\text{lab}(\psi)$ — and reports

$$a_x(\psi) = \frac{C\sin^2\theta_\text{lab}\cos 2\phi_\text{lab}}{A + B\sin^2\theta_\text{lab}}, \qquad a_y(\psi) = \frac{C\sin^2\theta_\text{lab}\sin 2\phi_\text{lab}}{A + B\sin^2\theta_\text{lab}}.$$

The cell below treats $(\Theta_\text{axis}, \Phi_\text{axis}, \theta_0, \phi_0)$ as a **4-parameter** input (no use of Λ or α anywhere) and animates one full rotation, showing four synchronised views:

1. **Unit sphere** — rotor as a 3D arrow tracing its circular orbit around $\hat{\mathbf k}$.
2. **Anisotropy plane** — the same motion projected onto $(a_x, a_y)$.
3. **$\theta_\text{lab}(\psi)$** — the polar angle in the lab frame.
4. **$\phi_\text{lab}(\psi)$** — the azimuth in the lab frame.

Markers every 10° in $\psi$ make the temporal correspondence explicit.

In [ ]:
# ── 4-parameter rotation demo: rotate m0 = (theta0, phi0) around k̂ by ψ ────
# All four parameters are independent here — no Λ/α anywhere.

# Motor axis (lab frame)
Theta_axis_anim = 0.0   # deg
Phi_axis_anim   = 0.0    # deg

# Initial rod orientation (lab frame), independent of axis
theta0_rod = 60.0   # deg — polar angle of m̂_0 from lab z
phi0_rod   = 30.0   # deg — azimuth of m̂_0 in lab

# Rotation sweep
N_anim = 360                          # 1° per frame
mark_step_deg = 10                    # static markers every 10°

# Reuse fourkas_ABC / theta_to_r from the early cells.
A_a, B_a, C_a = fourkas_ABC(np.arcsin(NA / nw))

def _rodrigues_R(k_hat, psi):
    """Rotation matrix about unit vector k_hat by angle psi."""
    K = np.array([[       0, -k_hat[2],  k_hat[1]],
                  [ k_hat[2],        0, -k_hat[0]],
                  [-k_hat[1],  k_hat[0],        0]])
    return np.eye(3) + np.sin(psi) * K + (1 - np.cos(psi)) * (K @ K)

# Build axis and initial rod from their (independent) lab-frame angles
_th_ax = np.radians(Theta_axis_anim); _ph_ax = np.radians(Phi_axis_anim)
k_anim = np.array([np.sin(_th_ax) * np.cos(_ph_ax),
                   np.sin(_th_ax) * np.sin(_ph_ax),
                   np.cos(_th_ax)])

_th0 = np.radians(theta0_rod); _ph0 = np.radians(phi0_rod)
m0   = np.array([np.sin(_th0) * np.cos(_ph0),
                 np.sin(_th0) * np.sin(_ph0),
                 np.cos(_th0)])

# Rotate m0 around k̂ for ψ in [0, 2π)
psi_anim   = np.linspace(0.0, 2 * np.pi, N_anim, endpoint=False)
m_traj     = np.array([_rodrigues_R(k_anim, p) @ m0 for p in psi_anim])  # (N, 3)
mx, my, mz = m_traj[:, 0], m_traj[:, 1], m_traj[:, 2]

# Lab-frame angles
theta_lab = np.arccos(np.clip(mz, -1.0, 1.0))
phi_lab   = np.arctan2(my, mx)

# Fourkas signal
r_lab = theta_to_r(theta_lab, A_a, B_a, C_a)
ax_sig = r_lab * np.cos(2 * phi_lab)
ay_sig = r_lab * np.sin(2 * phi_lab)

# 10° marker indices
mark_idx = np.arange(0, N_anim, max(1, int(round(mark_step_deg * N_anim / 360))))

# ── Stop any previous demo animation ──────────────────────────────────────────
_prev = globals().get('anim_demo')
if _prev is not None and hasattr(_prev, 'event_source') and _prev.event_source:
    _prev.event_source.stop()

# ── 4-panel figure: sphere | anisotropy / theta_lab | phi_lab ───────────────
fig_demo = plt.figure(num='rotation_demo', figsize=(12, 9))
fig_demo.clf()
gs = fig_demo.add_gridspec(2, 2)
ax_sph  = fig_demo.add_subplot(gs[0, 0], projection='3d')
ax_ani  = fig_demo.add_subplot(gs[0, 1])
ax_th   = fig_demo.add_subplot(gs[1, 0])
ax_ph   = fig_demo.add_subplot(gs[1, 1], sharex=ax_th)

# --- Sphere panel ---
# Faint trajectory and 10° markers
ax_sph.plot(mx, my, mz, '-', lw=0.6, color='0.6', alpha=0.4)
ax_sph.scatter(mx[mark_idx], my[mark_idx], mz[mark_idx],
               c=psi_anim[mark_idx] * 180 / np.pi, cmap='hsv',
               s=14, alpha=0.7, lw=0)
# Motor axis arrow
ax_sph.quiver(0, 0, 0, *k_anim, length=1.15, color='k',
              arrow_length_ratio=0.12, lw=2.0)
ax_sph.text(*(1.20 * k_anim), r'$\hat{\mathbf{k}}$', fontsize=11)
# Initial rod (psi = 0) — drawn as a yellow arrow from origin
ax_sph.quiver(0, 0, 0, m0[0], m0[1], m0[2],
              length=1.0, color='goldenrod', lw=2.0,
              arrow_length_ratio=0.15, alpha=0.9)
ax_sph.scatter([m0[0]], [m0[1]], [m0[2]],
               s=55, marker='s', c='yellow', edgecolors='k', lw=1.0, zorder=5)
ax_sph.text(*(1.10 * m0), r'$\hat{\mathbf{m}}_0$', fontsize=11, color='goldenrod')
# Animated rod arrow (replaced every frame) + leading dot
rod_quiver = [ax_sph.quiver(0, 0, 0, m0[0], m0[1], m0[2],
                            length=1.0, color='tab:red', lw=3.0,
                            arrow_length_ratio=0.18)]
rod_dot, = ax_sph.plot([m0[0]], [m0[1]], [m0[2]], 'o',
                       color='tab:red', ms=9, zorder=6)
# Animated label that follows the tip of m̂(ψ)
rod_label = ax_sph.text(1.10 * m0[0], 1.10 * m0[1], 1.10 * m0[2],
                        r'$\hat{\mathbf{m}}(\psi)$', fontsize=11,
                        color='tab:red')
ax_sph.set_xlim(-1, 1); ax_sph.set_ylim(-1, 1); ax_sph.set_zlim(-1, 1)
ax_sph.set_xlabel('x'); ax_sph.set_ylabel('y'); ax_sph.set_zlabel('z')
ax_sph.set_title(f'Rod $\\hat{{\\mathbf{{m}}}}(\\psi)$ on sphere\n'
                 f'axis ($\\Theta_a$={Theta_axis_anim}°, $\\Phi_a$={Phi_axis_anim}°)  '
                 f'$\\hat{{m}}_0$ ($\\theta_0$={theta0_rod}°, $\\phi_0$={phi0_rod}°)',
                 fontsize=10)

# --- Anisotropy panel ---
ax_ani.plot(np.append(ax_sig, ax_sig[0]),
            np.append(ay_sig, ay_sig[0]),
            '-', lw=0.7, color='0.7', alpha=0.7)
ax_ani.scatter(ax_sig[mark_idx], ay_sig[mark_idx],
               c=psi_anim[mark_idx] * 180 / np.pi, cmap='hsv',
               s=20, alpha=0.85, lw=0)
ax_ani.scatter(ax_sig[0], ay_sig[0], s=80, marker='s',
               c='yellow', edgecolors='k', lw=1.2, zorder=5,
               label=r'$\psi = 0$')
ani_dot, = ax_ani.plot([ax_sig[0]], [ay_sig[0]], 'o',
                       color='tab:red', ms=10, zorder=6)
ax_ani.axhline(0, color='grey', lw=0.5); ax_ani.axvline(0, color='grey', lw=0.5)
ax_ani.set_xlim(-1, 1); ax_ani.set_ylim(-1, 1); ax_ani.set_aspect('equal')
ax_ani.set_xlabel(r'$a_x$'); ax_ani.set_ylabel(r'$a_y$')
ax_ani.set_title('Anisotropy $(a_x, a_y)$  — coloured every 10°')
ax_ani.legend(fontsize=8, loc='upper right')
ax_ani.grid(alpha=0.3)

# --- theta_lab panel ---
psi_deg_anim = np.degrees(psi_anim)
ax_th.plot(psi_deg_anim, np.degrees(theta_lab), '-', color='tab:blue', lw=1.2)
ax_th.scatter(psi_deg_anim[mark_idx], np.degrees(theta_lab[mark_idx]),
              c=psi_deg_anim[mark_idx], cmap='hsv', s=18, lw=0)
th_dot, = ax_th.plot([0], [np.degrees(theta_lab[0])], 'o',
                     color='tab:red', ms=8, zorder=5)
ax_th.set_xlabel(r'$\psi$ (deg)'); ax_th.set_ylabel(r'$\theta_\text{lab}$ (deg)')
ax_th.set_title(r'Lab polar angle $\theta_\text{lab}(\psi)$')
ax_th.grid(alpha=0.3)

# --- phi_lab panel (unwrapped for readability) ---
phi_lab_uw = np.unwrap(phi_lab)
ax_ph.plot(psi_deg_anim, np.degrees(phi_lab_uw), '-', color='tab:green', lw=1.2)
ax_ph.scatter(psi_deg_anim[mark_idx], np.degrees(phi_lab_uw[mark_idx]),
              c=psi_deg_anim[mark_idx], cmap='hsv', s=18, lw=0)
ph_dot, = ax_ph.plot([0], [np.degrees(phi_lab_uw[0])], 'o',
                     color='tab:red', ms=8, zorder=5)
ax_ph.set_xlabel(r'$\psi$ (deg)'); ax_ph.set_ylabel(r'$\phi_\text{lab}$ (deg, unwrapped)')
ax_ph.set_title(r'Lab azimuth $\phi_\text{lab}(\psi)$')
ax_ph.grid(alpha=0.3)

# ── Animation update ─────────────────────────────────────────────────────────
def _update_demo(frame):
    # Replace 3D quiver (cannot be reused efficiently; remove + redraw)
    rod_quiver[0].remove()
    rod_quiver[0] = ax_sph.quiver(0, 0, 0,
                                  mx[frame], my[frame], mz[frame],
                                  length=1.0, color='tab:red', lw=3.0,
                                  arrow_length_ratio=0.18)
    rod_dot.set_data([mx[frame]], [my[frame]])
    rod_dot.set_3d_properties([mz[frame]])
    rod_label.set_position((1.10 * mx[frame], 1.10 * my[frame]))
    rod_label.set_3d_properties(1.10 * mz[frame], zdir=None)

    ani_dot.set_data([ax_sig[frame]], [ay_sig[frame]])
    th_dot.set_data([psi_deg_anim[frame]], [np.degrees(theta_lab[frame])])
    ph_dot.set_data([psi_deg_anim[frame]], [np.degrees(phi_lab_uw[frame])])

    fig_demo.suptitle(f'$\\psi = {psi_deg_anim[frame]:6.1f}°$    '
                      f'$\\theta_\\text{{lab}} = {np.degrees(theta_lab[frame]):.2f}°$    '
                      f'$\\phi_\\text{{lab}} = {np.degrees(phi_lab[frame]):.2f}°$    '
                      f'$(a_x, a_y) = ({ax_sig[frame]:+.3f}, {ay_sig[frame]:+.3f})$',
                      fontsize=11)
    return rod_dot, ani_dot, th_dot, ph_dot

anim_demo = FuncAnimation(fig_demo, _update_demo, frames=N_anim,
                          interval=40, blit=False)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

## How the 4-parameter input collapses to 3 observable parameters

The previous cell took **four** independent inputs — $(\Theta_\text{axis}, \Phi_\text{axis})$ for the axis and $(\theta_0, \phi_0)$ for the rod's initial direction. But the observable $(a_x, a_y)$ loop only depends on **three** combinations of them:

1. $\Theta_\text{axis}, \Phi_\text{axis}$ — the lab-frame direction of $\hat{\mathbf k}$ (2 numbers).
2. $\Lambda \equiv \arccos(\hat{\mathbf m}_0 \cdot \hat{\mathbf k})$ — the angle between the rod and the axis (1 number, latitude on the cone).

The 4th would-be parameter is the **azimuth of $\hat{\mathbf m}_0$ around $\hat{\mathbf k}$**:
$\alpha \equiv$ angle of $\hat{\mathbf m}_0$ measured around the axis from some reference direction in the plane perpendicular to $\hat{\mathbf k}$. As motor phase $\psi$ advances, $\alpha \to \alpha + \psi$ — so $\alpha$ is mathematically indistinguishable from a redefinition of the zero of $\psi$. **Same loop, different label on the starting point.**

The cell below verifies this directly:

- Pick three different $(\theta_0, \phi_0)$ values, all chosen so that $\hat{\mathbf m}_0$ makes the **same** angle $\Lambda$ with $\hat{\mathbf k}$ (i.e. they sit on the same latitude on the cone, just at different azimuths around it).
- Each is rotated around $\hat{\mathbf k}$ by $\psi \in [0, 2\pi)$ and the $(a_x, a_y)$ loop is computed.
- The three loops overlap exactly; only the position of the $\psi=0$ marker (yellow square) differs.

Then a final loop is added with a **different $\Lambda$** to show that *changing $\Lambda$* really does deform the loop. So $\Lambda$ is the genuine 3rd geometric DOF; $\alpha$ is a phase reference.

In [ ]:
# ── Three rod starts at the same Λ, then a 4th at a different Λ ──────────────

# Same axis as the animation cell above (re-used for visual continuity).
Theta_axis_c = Theta_axis_anim
Phi_axis_c   = Phi_axis_anim

th_ax_c = np.radians(Theta_axis_c); ph_ax_c = np.radians(Phi_axis_c)
k_c = np.array([np.sin(th_ax_c) * np.cos(ph_ax_c),
                np.sin(th_ax_c) * np.sin(ph_ax_c),
                np.cos(th_ax_c)])

# Build an orthonormal basis (k̂, e1, e2) used only to construct test inputs.
if abs(k_c[2]) < 0.9999:
    e1_c = np.array([0.0, 0.0, 1.0]) - k_c[2] * k_c
    e1_c /= np.linalg.norm(e1_c)
else:
    e1_c = np.array([1.0, 0.0, 0.0])
e2_c = np.cross(k_c, e1_c)

def _m0_from_lambda_alpha(Lambda_deg, alpha_deg):
    """Construct an initial rod direction at polar angle Λ from k̂ and azimuth α
    around k̂. Returned in lab Cartesian coords."""
    L = np.radians(Lambda_deg); a = np.radians(alpha_deg)
    return np.cos(L) * k_c + np.sin(L) * (np.cos(a) * e1_c + np.sin(a) * e2_c)

def _loop(m0_vec, N=N_anim):
    """Full (a_x, a_y) loop for an initial rod direction m0_vec."""
    psi = np.linspace(0.0, 2 * np.pi, N, endpoint=False)
    traj = np.array([_rodrigues_R(k_c, p) @ m0_vec for p in psi])
    th_lab = np.arccos(np.clip(traj[:, 2], -1, 1))
    ph_lab = np.arctan2(traj[:, 1], traj[:, 0])
    r = theta_to_r(th_lab, A_a, B_a, C_a)
    return r * np.cos(2 * ph_lab), r * np.sin(2 * ph_lab), traj

# Three different starting azimuths α at fixed Λ
Lambda_c   = 35.0
alpha_list = [0.0, 90.0, 215.0]
runs_same = []
for a_deg in alpha_list:
    m0v = _m0_from_lambda_alpha(Lambda_c, a_deg)
    # Recover the lab-frame (theta0, phi0) of this m0 — the *real* free numbers.
    th0_lab = float(np.degrees(np.arccos(np.clip(m0v[2], -1, 1))))
    ph0_lab = float(np.degrees(np.arctan2(m0v[1], m0v[0])))
    ax_l, ay_l, traj_l = _loop(m0v)
    runs_same.append({'alpha': a_deg, 'theta0_lab': th0_lab, 'phi0_lab': ph0_lab,
                      'ax': ax_l, 'ay': ay_l, 'm0': m0v, 'traj': traj_l})

# A 4th run with a different Λ to show shape change
Lambda_diff = 55.0
m0_diff = _m0_from_lambda_alpha(Lambda_diff, 0.0)
ax_d, ay_d, traj_d = _loop(m0_diff)
th0_diff = float(np.degrees(np.arccos(np.clip(m0_diff[2], -1, 1))))
ph0_diff = float(np.degrees(np.arctan2(m0_diff[1], m0_diff[0])))

# ── Plot ─────────────────────────────────────────────────────────────────────
fig_col = plt.figure(num='collapse_demo', figsize=(12, 5.5))
fig_col.clf()
ax_sp = fig_col.add_subplot(1, 2, 1, projection='3d')
ax_an = fig_col.add_subplot(1, 2, 2)

colors = ['tab:blue', 'tab:orange', 'tab:purple']

# Sphere panel
ax_sp.quiver(0, 0, 0, *k_c, length=1.15, color='k', lw=2.0,
             arrow_length_ratio=0.12, label=r'$\hat{\mathbf{k}}$')
for run, c in zip(runs_same, colors):
    ax_sp.plot(run['traj'][:, 0], run['traj'][:, 1], run['traj'][:, 2],
               '-', color=c, lw=1.0, alpha=0.7,
               label=rf"$\Lambda$={Lambda_c}° α={run['alpha']:.0f}°  "
                     rf"($\theta_0$={run['theta0_lab']:.1f}°, $\phi_0$={run['phi0_lab']:.1f}°)")
    # m̂_0 as an arrow from origin
    ax_sp.quiver(0, 0, 0, run['m0'][0], run['m0'][1], run['m0'][2],
                 length=1.0, color=c, lw=2.0, arrow_length_ratio=0.15)
    ax_sp.scatter([run['m0'][0]], [run['m0'][1]], [run['m0'][2]],
                  s=70, marker='s', c=c, edgecolors='k', lw=1.0, zorder=6)
ax_sp.plot(traj_d[:, 0], traj_d[:, 1], traj_d[:, 2], '-',
           color='tab:red', lw=1.2, alpha=0.9,
           label=rf'$\Lambda$={Lambda_diff}° (different latitude)')
ax_sp.quiver(0, 0, 0, m0_diff[0], m0_diff[1], m0_diff[2],
             length=1.0, color='tab:red', lw=2.0, arrow_length_ratio=0.15)
ax_sp.scatter([m0_diff[0]], [m0_diff[1]], [m0_diff[2]],
              s=70, marker='s', c='tab:red', edgecolors='k', lw=1.0, zorder=6)
ax_sp.set_xlim(-1, 1); ax_sp.set_ylim(-1, 1); ax_sp.set_zlim(-1, 1)
ax_sp.set_xlabel('x'); ax_sp.set_ylabel('y'); ax_sp.set_zlabel('z')
ax_sp.set_title('Sphere — same Λ trajectories share one circle')
ax_sp.legend(fontsize=7, loc='upper left')

# Anisotropy panel — same Λ runs overlap exactly (slightly offset linewidths so all show)
for run, c, lw in zip(runs_same, colors, [4.0, 2.4, 1.0]):
    ax_an.plot(np.append(run['ax'], run['ax'][0]),
               np.append(run['ay'], run['ay'][0]),
               '-', color=c, lw=lw, alpha=0.85,
               label=rf"$\Lambda$={Lambda_c}° α={run['alpha']:.0f}°")
    ax_an.scatter(run['ax'][0], run['ay'][0], s=80, marker='s',
                  c=c, edgecolors='k', lw=1.0, zorder=5)
ax_an.plot(np.append(ax_d, ax_d[0]), np.append(ay_d, ay_d[0]), '-',
           color='tab:red', lw=1.6, alpha=0.85,
           label=rf'$\Lambda$={Lambda_diff}° (different shape)')
ax_an.scatter(ax_d[0], ay_d[0], s=80, marker='s',
              c='tab:red', edgecolors='k', lw=1.0, zorder=5)
ax_an.axhline(0, color='grey', lw=0.5); ax_an.axvline(0, color='grey', lw=0.5)
ax_an.set_xlim(-1, 1); ax_an.set_ylim(-1, 1); ax_an.set_aspect('equal')
ax_an.set_xlabel(r'$a_x$'); ax_an.set_ylabel(r'$a_y$')
ax_an.set_title('Anisotropy — same Λ ⇒ same loop, only ψ=0 (square) differs')
ax_an.grid(alpha=0.3); ax_an.legend(fontsize=8, loc='upper right')

plt.tight_layout()
plt.show()

# ── Numerical confirmation: pairwise loop distance after phase alignment ─────
print('Numerical check — same-Λ runs (after rotating each loop by its own α):')
print(f'{"α (deg)":>10} | {"(θ0, φ0) lab (deg)":>22} | {"max |loop − ref|":>18}')
print('-' * 60)
ax_ref = runs_same[0]['ax']; ay_ref = runs_same[0]['ay']
for run in runs_same:
    shift = int(round(-run['alpha'] / 360.0 * N_anim)) % N_anim
    ax_s = np.roll(run['ax'], shift); ay_s = np.roll(run['ay'], shift)
    d_max = float(np.max(np.hypot(ax_s - ax_ref, ay_s - ay_ref)))
    print(f"  {run['alpha']:8.1f} | "
          f"({run['theta0_lab']:6.1f}, {run['phi0_lab']:6.1f})  | "
          f"{d_max:18.2e}")

print('\n→ Three different (θ0, φ0) inputs collapse to the same loop, '
      'shifted only in ψ.')
print(f'   Different-Λ run (Λ={Lambda_diff}° at θ0={th0_diff:.1f}°, '
      f'φ0={ph0_diff:.1f}°): visibly different loop ⇒ Λ is a real DOF.')

## Multiple rods clamped to the same rotor — intensity-sum model

Several rods are rigidly clamped to the same rotor and rotate together around
the lab-frame axis $\hat{\mathbf k}$ (specified by $\Theta_{\text{axis}}, \Phi_{\text{axis}}$).
Each rod $i$ has its own cone angle $\Lambda_i$ (rod-to-axis polar angle) and a
phase offset $\Delta\psi_i$ around the axis (so rod 0 sits at the reference
phase, $\Delta\psi_0 = 0$, and the others are shifted in their azimuth around
$\hat{\mathbf k}$). At motor phase $\psi$ the rod direction is

$$\hat{\mathbf m}_i(\psi) = \cos\Lambda_i\,\hat{\mathbf k} + \sin\Lambda_i\,\bigl[\cos(\psi+\Delta\psi_i)\,\hat{\mathbf e}_1 + \sin(\psi+\Delta\psi_i)\,\hat{\mathbf e}_2\bigr].$$

The Fourkas signal in each polarisation channel is

$$\begin{aligned}
I_0    &= A + B\sin^2\theta + C\sin^2\theta\,\cos 2\phi\\
I_{45} &= A + B\sin^2\theta + C\sin^2\theta\,\sin 2\phi\\
I_{90} &= A + B\sin^2\theta - C\sin^2\theta\,\cos 2\phi\\
I_{135}&= A + B\sin^2\theta - C\sin^2\theta\,\sin 2\phi.
\end{aligned}$$

For multiple rods of equal brightness the **intensities add**, so the combined
detector counts are $I^{\text{tot}}_c = \sum_i I_c(\theta_{\text{lab},i}, \phi_{\text{lab},i})$, and the
anisotropies are

$$a_x^{\text{tot}} = \frac{I_0^{\text{tot}}-I_{90}^{\text{tot}}}{I_0^{\text{tot}}+I_{90}^{\text{tot}}},\qquad a_y^{\text{tot}} = \frac{I_{45}^{\text{tot}}-I_{135}^{\text{tot}}}{I_{45}^{\text{tot}}+I_{135}^{\text{tot}}}.$$

Below: pick a few configurations by hand and compare to a single-rod baseline.

In [ ]:
# ── Multi-rod via 4-channel intensity sum ────────────────────────────────────
# All four input numbers per rod live here:
#   - Theta_axis, Phi_axis       → axis direction in the lab
#   - rod_list                   → list of (Lambda_deg, dpsi_deg) pairs
#     where dpsi is the phase offset around Ķ relative to rod 0
#     (by convention rod_list[0] should have dpsi = 0)

Theta_axis_mr = 15.0
Phi_axis_mr   = 0.0

# A few hand-picked configurations to try — edit freely.
# Each rod entry is (Lambda_deg, dpsi_deg, brightness).  Brightness is
# a relative prefactor multiplying that rod's 4-channel intensities
# before summing, so e.g. a value of 1.5 means rod 2 emits 1.5x rod 1.
configs = {
    'single rod':
        [(35.0, 0.0, 1.0)],
    'two rods (equal brightness)':
        [(60.0, 0.0, 1.0), (35.0, 15.0, 1.0)],
    'two rods (rod 2 = 1.5x rod 1)':
        [(60.0, 0.0, 1.0), (35.0, 15.0, 1.5)],
}

N_psi_mr = 720
psi_mr = np.linspace(0.0, 2 * np.pi, N_psi_mr, endpoint=False)

# Axis frame (k̂, e1, e2)
_th_ax = np.radians(Theta_axis_mr); _ph_ax = np.radians(Phi_axis_mr)
k_mr = np.array([np.sin(_th_ax) * np.cos(_ph_ax),
                 np.sin(_th_ax) * np.sin(_ph_ax),
                 np.cos(_th_ax)])
if abs(k_mr[2]) < 0.9999:
    e1_mr = np.array([0.0, 0.0, 1.0]) - k_mr[2] * k_mr
    e1_mr /= np.linalg.norm(e1_mr)
else:
    e1_mr = np.array([1.0, 0.0, 0.0])
e2_mr = np.cross(k_mr, e1_mr)


def _per_rod_intensities(Lambda_deg, dpsi_deg, brightness=1.0):
    """Return (I0, I45, I90, I135) of shape (N_psi,) for one rod, scaled by brightness."""
    L     = np.radians(Lambda_deg)
    phase = psi_mr + np.radians(dpsi_deg)
    m = (np.cos(L) * k_mr[None, :]
         + np.sin(L) * (np.cos(phase)[:, None] * e1_mr[None, :]
                        + np.sin(phase)[:, None] * e2_mr[None, :]))
    th_lab = np.arccos(np.clip(m[:, 2], -1.0, 1.0))
    ph_lab = np.arctan2(m[:, 1], m[:, 0])
    s2     = np.sin(th_lab) ** 2
    base   = A_a + B_a * s2
    cos2   = C_a * s2 * np.cos(2 * ph_lab)
    sin2   = C_a * s2 * np.sin(2 * ph_lab)
    return (brightness * (base + cos2),
            brightness * (base + sin2),
            brightness * (base - cos2),
            brightness * (base - sin2))


def _combined_anisotropy(rod_list):
    """Sum 4-channel intensities over all rods, then form ax, ay.

    rod_list entries may be (Lambda, dpsi) or (Lambda, dpsi, brightness);
    missing brightness defaults to 1.0.
    """
    I0 = I45 = I90 = I135 = 0.0
    for entry in rod_list:
        if len(entry) == 2:
            L_deg, dpsi_deg = entry; b = 1.0
        else:
            L_deg, dpsi_deg, b = entry
        i0, i45, i90, i135 = _per_rod_intensities(L_deg, dpsi_deg, b)
        I0 += i0; I45 += i45; I90 += i90; I135 += i135
    ax = (I0  - I90 ) / (I0  + I90 )
    ay = (I45 - I135) / (I45 + I135)
    return ax, ay


# ── Plot every configuration ────────────────────────────────────────────────
# Close any previous figure with this name so the subplot grid is rebuilt
# from scratch when len(configs) changes between runs.
if plt.fignum_exists('multirod_intensity'):
    plt.close('multirod_intensity')
n_cfg = len(configs)
fig_mr, axes_mr = plt.subplots(1, n_cfg, num='multirod_intensity',
                               figsize=(4.2 * n_cfg, 4.6),
                               sharex=True, sharey=True, squeeze=False)
axes_mr = axes_mr.ravel()

for ax_panel, (label, rod_list) in zip(axes_mr, configs.items()):
    # Faint per-rod loops (brightness ignored here so shape is visible)
    for entry in rod_list:
        L_deg, dpsi_deg = entry[0], entry[1]
        ax_one, ay_one = _combined_anisotropy([(L_deg, dpsi_deg, 1.0)])
        ax_panel.plot(np.append(ax_one, ax_one[0]),
                      np.append(ay_one, ay_one[0]),
                      '-', color='0.7', lw=0.7, alpha=0.8)
    # Combined loop in red
    ax_tot, ay_tot = _combined_anisotropy(rod_list)
    ax_panel.plot(np.append(ax_tot, ax_tot[0]),
                  np.append(ay_tot, ay_tot[0]),
                  '-', color='tab:red', lw=2.0)
    rms = float(np.sqrt(np.mean(ax_tot ** 2 + ay_tot ** 2)))
    ax_panel.text(0.04, 0.96, f'rms = {rms:.3f}',
                  transform=ax_panel.transAxes, fontsize=8,
                  va='top', ha='left',
                  bbox=dict(boxstyle='round,pad=0.2', fc='white',
                            ec='0.7', lw=0.5, alpha=0.9))
    def _fmt(entry):
        L, d = entry[0], entry[1]
        b = entry[2] if len(entry) > 2 else 1.0
        return f'Λ={L:.0f}°, Δψ={d:+.0f}°, I={b:g}'
    cfg_text = '\n'.join(_fmt(e) for e in rod_list)
    ax_panel.text(0.04, 0.04, cfg_text,
                  transform=ax_panel.transAxes, fontsize=7,
                  va='bottom', ha='left',
                  bbox=dict(boxstyle='round,pad=0.2', fc='white',
                            ec='0.7', lw=0.5, alpha=0.9))
    ax_panel.axhline(0, color='grey', lw=0.4); ax_panel.axvline(0, color='grey', lw=0.4)
    ax_panel.set_xlim(-1, 1); ax_panel.set_ylim(-1, 1); ax_panel.set_aspect('equal')
    ax_panel.set_title(label, fontsize=10)
    ax_panel.set_xlabel(r'$a_x$')
axes_mr[0].set_ylabel(r'$a_y$')
fig_mr.suptitle(rf'Multi-rod (intensity sum), axis $\Theta_a$={Theta_axis_mr}°, '
                rf'$\Phi_a$={Phi_axis_mr}°    grey: per-rod   red: combined',
                fontsize=11)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()


In [ ]:
# ── Sweep: many (theta_axis, two-rod) configurations ─────────────────────
# Reuses A_a, B_a, C_a from earlier cells. Builds its own axis frame and
# psi grid for each (Theta_axis, rod_list) so it does not depend on the
# globals (Theta_axis_mr, k_mr, psi_mr, ...) from the previous cell.

N_psi_sw = 720
psi_sw   = np.linspace(0.0, 2 * np.pi, N_psi_sw, endpoint=False)


def _axis_frame(Theta_axis_deg, Phi_axis_deg=0.0):
    th = np.radians(Theta_axis_deg); ph = np.radians(Phi_axis_deg)
    k  = np.array([np.sin(th) * np.cos(ph),
                   np.sin(th) * np.sin(ph),
                   np.cos(th)])
    if abs(k[2]) < 0.9999:
        e1 = np.array([0.0, 0.0, 1.0]) - k[2] * k
        e1 /= np.linalg.norm(e1)
    else:
        e1 = np.array([1.0, 0.0, 0.0])
    e2 = np.cross(k, e1)
    return k, e1, e2


def _two_rod_loop(Theta_axis_deg, rod_list, Phi_axis_deg=0.0):
    """Return ax(psi), ay(psi) for the intensity-summed multi-rod model."""
    k, e1, e2 = _axis_frame(Theta_axis_deg, Phi_axis_deg)
    I0 = I45 = I90 = I135 = 0.0
    for L_deg, dpsi_deg, b in rod_list:
        L     = np.radians(L_deg)
        phase = psi_sw + np.radians(dpsi_deg)
        m = (np.cos(L) * k[None, :]
             + np.sin(L) * (np.cos(phase)[:, None] * e1[None, :]
                            + np.sin(phase)[:, None] * e2[None, :]))
        th_lab = np.arccos(np.clip(m[:, 2], -1.0, 1.0))
        ph_lab = np.arctan2(m[:, 1], m[:, 0])
        s2   = np.sin(th_lab) ** 2
        base = A_a + B_a * s2
        cos2 = C_a * s2 * np.cos(2 * ph_lab)
        sin2 = C_a * s2 * np.sin(2 * ph_lab)
        I0   += b * (base + cos2)
        I45  += b * (base + sin2)
        I90  += b * (base - cos2)
        I135 += b * (base - sin2)
    return (I0 - I90) / (I0 + I90), (I45 - I135) / (I45 + I135)


# ── Sweep grid: theta_axis along columns, configurations along rows ───────────
theta_axis_sweep = [0.0, 10.0, 25.0, 45.0, 70.0, 90.0]

# Each row is one (Lambda1, dpsi1, b1, Lambda2, dpsi2, b2) two-rod config.
rod_configs_sweep = [
    ("equal, opposite",          35.0,   0.0, 1.0,  35.0, 180.0, 1.0),
    ("equal, perpendicular",     35.0,   0.0, 1.0,  35.0,  90.0, 1.0),
    ("diff Λ, equal bright",      30.0,   0.0, 1.0,  60.0,  45.0, 1.0),
    ("diff Λ, rod2 = 2x rod1",    30.0,   0.0, 1.0,  60.0,  45.0, 2.0),
    ("diff Λ, rod2 = 0.3x rod1",  30.0,   0.0, 1.0,  60.0,  45.0, 0.3),
]

n_rows = len(rod_configs_sweep)
n_cols = len(theta_axis_sweep)

if plt.fignum_exists("multirod_sweep"):
    plt.close("multirod_sweep")
fig_sw, axes_sw = plt.subplots(n_rows, n_cols, num="multirod_sweep",
                               figsize=(2.4 * n_cols, 2.4 * n_rows),
                               sharex=True, sharey=True, squeeze=False)

for i, (label, L1, d1, b1, L2, d2, b2) in enumerate(rod_configs_sweep):
    rod_list = [(L1, d1, b1), (L2, d2, b2)]
    for j, th_ax in enumerate(theta_axis_sweep):
        ax = axes_sw[i, j]
        # Faint per-rod loops (brightness=1 for shape only)
        for L, d, _ in rod_list:
            ax_one, ay_one = _two_rod_loop(th_ax, [(L, d, 1.0)])
            ax.plot(np.append(ax_one, ax_one[0]),
                    np.append(ay_one, ay_one[0]),
                    "-", color="0.75", lw=0.5, alpha=0.7)
        # Combined loop with the actual brightnesses
        ax_tot, ay_tot = _two_rod_loop(th_ax, rod_list)
        ax.plot(np.append(ax_tot, ax_tot[0]),
                np.append(ay_tot, ay_tot[0]),
                "-", color="tab:red", lw=1.4)
        ax.axhline(0, color="grey", lw=0.3); ax.axvline(0, color="grey", lw=0.3)
        ax.set_xlim(-1, 1); ax.set_ylim(-1, 1); ax.set_aspect("equal")
        ax.tick_params(labelsize=7)
        if i == 0:
            ax.set_title(rf"$\Theta_a$={th_ax:.0f}°", fontsize=9)
        if j == 0:
            ax.set_ylabel(label + "\n" +
                          rf"$\Lambda_1$={L1:.0f}°,$\Delta\psi_1$={d1:+.0f}°,I={b1:g}" + "\n" +
                          rf"$\Lambda_2$={L2:.0f}°,$\Delta\psi_2$={d2:+.0f}°,I={b2:g}",
                          fontsize=7)
        if i == n_rows - 1:
            ax.set_xlabel(r"$a_x$", fontsize=8)

fig_sw.suptitle("Two-rod sweep — columns: $\\Theta_\\mathrm{axis}$,  rows: rod config   "
                "(grey: per-rod, red: combined)", fontsize=11)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()


## Origami hinge — partial-arc rotation near $\Lambda=90°$

A DNA origami hinge has one arm fixed to the substrate and the other free to swing
between two stops. A rod glued to the moving arm therefore sweeps only a **partial
arc** in $\psi$, not a full $2\pi$ revolution. Choosing the hinge axis horizontal in
the lab corresponds to $\Theta_\text{axis}\approx 90°$, and a rod that lies in
the plane swept by the arm has $\Lambda\approx 90°$ — small misalignments between
arm and rod show up as small departures of $\Lambda$ from $90°$.

Below: same intensity-sum machinery as before, but $\psi$ runs only from $0$ to
$\psi_\max$ (e.g. $\pi/3$). Try one rod and two rods with similar $\Lambda\approx 90°$
and a small $\Delta\psi$ lag between them.

In [ ]:
# ── Origami hinge: partial-arc sweep near Lambda = 90° ──────────────────
# Reuses _axis_frame, A_a, B_a, C_a from earlier cells.

Theta_axis_hg = 90.0     # hinge axis in lab plane
Phi_axis_hg   = 0.0
# Arm sweeps so the rod's lab polar angle goes from psi_min_deg to psi_max_deg.
# psi=90° means the rod lies flat on the substrate (theta_lab=90°);
# psi=30° means it's tilted up to theta_lab=30° from the vertical.
psi_min_deg   = 30.0
psi_max_deg   = 90.0
N_psi_hg      = 400

psi_hg = np.linspace(np.radians(psi_min_deg), np.radians(psi_max_deg), N_psi_hg)


def _hinge_loop(rod_list, Theta_axis_deg=Theta_axis_hg,
                Phi_axis_deg=Phi_axis_hg, psi_grid=psi_hg):
    """Same intensity-sum model as before but on an arbitrary psi grid."""
    k, e1, e2 = _axis_frame(Theta_axis_deg, Phi_axis_deg)
    I0 = I45 = I90 = I135 = 0.0
    for L_deg, dpsi_deg, b in rod_list:
        L     = np.radians(L_deg)
        phase = psi_grid + np.radians(dpsi_deg)
        m = (np.cos(L) * k[None, :]
             + np.sin(L) * (np.cos(phase)[:, None] * e1[None, :]
                            + np.sin(phase)[:, None] * e2[None, :]))
        th_lab = np.arccos(np.clip(m[:, 2], -1.0, 1.0))
        ph_lab = np.arctan2(m[:, 1], m[:, 0])
        s2   = np.sin(th_lab) ** 2
        base = A_a + B_a * s2
        cos2 = C_a * s2 * np.cos(2 * ph_lab)
        sin2 = C_a * s2 * np.sin(2 * ph_lab)
        I0   += b * (base + cos2)
        I45  += b * (base + sin2)
        I90  += b * (base - cos2)
        I135 += b * (base - sin2)
    return (I0 - I90) / (I0 + I90), (I45 - I135) / (I45 + I135)


# A few hinge configurations — edit freely.
hinge_configs = {
    "single rod, Λ=90°":
        [(90.0, 0.0, 1.0)],
    "single rod, Λ=85° (small misalign)":
        [(85.0, 0.0, 1.0)],
    "two rods, Λ=90°,90°, Δψ=5°":
        [(90.0, 0.0, 1.0), (90.0, 5.0, 1.0)],
    "two rods, Λ=88°,92°, Δψ=10°":
        [(88.0, 0.0, 1.0), (92.0, 10.0, 1.0)],
    "single rod, Λ=75°":
        [(75.0, 0.0, 1.0)],
    "two rods, Λ=90°,75°, Δψ=10°":
        [(90.0, 0.0, 1.0), (75.0, 10.0, 1.0)],
}

n_h = len(hinge_configs)
if plt.fignum_exists("origami_hinge"):
    plt.close("origami_hinge")
fig_h, axes_h = plt.subplots(1, n_h, num="origami_hinge",
                             figsize=(3.8 * n_h, 4.2),
                             sharex=True, sharey=True, squeeze=False)
axes_h = axes_h.ravel()

for ax, (label, rod_list) in zip(axes_h, hinge_configs.items()):
    # Per-rod faint arcs (brightness = 1 for shape only)
    for L, d, _ in rod_list:
        ax_one, ay_one = _hinge_loop([(L, d, 1.0)])
        ax.plot(ax_one, ay_one, "-", color="0.75", lw=0.7, alpha=0.8)
    # Combined arc
    ax_tot, ay_tot = _hinge_loop(rod_list)
    sc = ax.scatter(ax_tot, ay_tot, c=np.degrees(psi_hg),
                    cmap="viridis", s=8, lw=0)
    # Mark endpoints
    ax.scatter(ax_tot[0],  ay_tot[0],  s=70, marker="s",
               c="yellow",  edgecolors="k", lw=1.0, zorder=6,
               label=rf"$\psi={psi_min_deg:.0f}°$")
    ax.scatter(ax_tot[-1], ay_tot[-1], s=70, marker="^",
               c="red",     edgecolors="k", lw=1.0, zorder=6,
               label=rf"$\psi={psi_max_deg:.0f}°$")
    cfg_text = "\n".join(f"Λ={L:.0f}°, Δψ={d:+.0f}°, I={b:g}"
                          for L, d, b in rod_list)
    ax.text(0.04, 0.04, cfg_text, transform=ax.transAxes, fontsize=7,
            va="bottom", ha="left",
            bbox=dict(boxstyle="round,pad=0.2", fc="white",
                      ec="0.7", lw=0.5, alpha=0.9))
    ax.axhline(0, color="grey", lw=0.4); ax.axvline(0, color="grey", lw=0.4)
    ax.set_xlim(-1, 1); ax.set_ylim(-1, 1); ax.set_aspect("equal")
    ax.set_title(label, fontsize=9)
    ax.set_xlabel(r"$a_x$")
    ax.legend(fontsize=7, loc="upper right")

axes_h[0].set_ylabel(r"$a_y$")
fig_h.colorbar(sc, ax=axes_h.tolist(), shrink=0.8, label=r"$\psi$ (deg)")
fig_h.suptitle(rf"Origami hinge — partial arc, axis $\Theta_a$={Theta_axis_hg:.0f}°, "
               rf"sweep {psi_min_deg:.0f}→{psi_max_deg:.0f}°",
               fontsize=11)
plt.show()


## Recovering $\theta_\text{lab}$ from the combined signal — when does the single-rod inversion fail?

The hinge sweep above gives, for each configuration, a combined $(a_x, a_y)$ trace.
If we **pretend** the trace came from a single rod, we can invert via Fourkas:

$$r(\psi) = \sqrt{a_x^2 + a_y^2}, \qquad \theta_\text{rec}(\psi) = \arcsin\sqrt{\frac{r\,A}{C - r\,B}}.$$

For a true single rod with $\Lambda=90°$ on a hinge axis at $\Theta_a=90°$, this returns
exactly $\theta_\text{lab}(\psi)=\psi$, so the recovered span equals the arm sweep
($\psi_\max-\psi_\min = 60°$). For multi-rod configurations the combined trace is **not**
a single-rod signal, and the recovered $\theta$ span generally drifts away from $60°$ —
more so as the rods differ more in $\Lambda$ or brightness.

Below: a wider zoo of 1/2/3-rod configurations, the recovered $\theta_\text{rec}(\psi)$
overlaid on the true $\psi$-line, and a bar-chart summary of the recovered span vs $60°$.

In [ ]:
# ── Single-rod theta recovery applied to multi-rod hinge signals ──────────
# Reuses _hinge_loop, psi_hg, psi_min_deg, psi_max_deg, A_a, B_a, C_a, r_to_theta.

# Expanded zoo — original 6, plus extra 2-rod (bigger swings) and 3-rod cases.
hinge_configs_zoo = {
    "1 rod  Λ=90":              [(90.0, 0.0, 1.0)],
    "1 rod  Λ=85":              [(85.0, 0.0, 1.0)],
    "1 rod  Λ=75":              [(75.0, 0.0, 1.0)],
    "1 rod  Λ=60":              [(60.0, 0.0, 1.0)],
    "2 rod  90/90  Δψ=5":       [(90.0, 0.0, 1.0), (90.0,  5.0, 1.0)],
    "2 rod  88/92  Δψ=10":      [(88.0, 0.0, 1.0), (92.0, 10.0, 1.0)],
    "2 rod  90/75  Δψ=10":      [(90.0, 0.0, 1.0), (75.0, 10.0, 1.0)],
    "2 rod  90/60  Δψ=20":      [(90.0, 0.0, 1.0), (60.0, 20.0, 1.0)],
    "2 rod  90/45  Δψ=30":      [(90.0, 0.0, 1.0), (45.0, 30.0, 1.0)],
    "2 rod  90/60  Δψ=20  I=3": [(90.0, 0.0, 1.0), (60.0, 20.0, 3.0)],
    "2 rod  90/60  Δψ=20  I=0.3":[(90.0, 0.0, 1.0),(60.0, 20.0, 0.3)],
    "3 rod  90/85/95  small":   [(90.0, 0.0, 1.0), (85.0, 5.0, 1.0), (95.0, -5.0, 1.0)],
    "3 rod  90/75/60  spread":  [(90.0, 0.0, 1.0), (75.0,10.0, 1.0), (60.0, 20.0, 1.0)],
    "3 rod  90/60/30  big":     [(90.0, 0.0, 1.0), (60.0,15.0, 1.0), (30.0, 30.0, 1.0)],
    "3 rod  90/60/30  bright2":[(90.0, 0.0, 1.0), (60.0,15.0, 2.0), (30.0, 30.0, 0.5)],
}

psi_deg_hg     = np.degrees(psi_hg)
true_span_deg  = psi_max_deg - psi_min_deg     # = 60° here

# True theta_lab(psi) for one rod on this hinge axis (Theta_axis_hg, Phi_axis_hg)
def _theta_lab_one(L_deg, dpsi_deg):
    k, e1, e2 = _axis_frame(Theta_axis_hg, Phi_axis_hg)
    L = np.radians(L_deg)
    phase = psi_hg + np.radians(dpsi_deg)
    m = (np.cos(L) * k[None, :]
         + np.sin(L) * (np.cos(phase)[:, None] * e1[None, :]
                        + np.sin(phase)[:, None] * e2[None, :]))
    return np.degrees(np.arccos(np.clip(m[:, 2], -1.0, 1.0)))

# Compute recovered theta for each config
results = []
for label, rod_list in hinge_configs_zoo.items():
    ax_t, ay_t = _hinge_loop(rod_list)
    r          = np.sqrt(ax_t**2 + ay_t**2)
    theta_rec  = np.degrees(r_to_theta(r, A_a, B_a, C_a))
    span_rec   = float(theta_rec.max() - theta_rec.min())
    # "True" reference: brightness-weighted average of per-rod theta_lab(psi).
    # For a single rod this is just its own theta_lab(psi); for multiple rods
    # it's the (heuristic) target a single-rod inversion would have to match.
    th_per = np.array([_theta_lab_one(L, d) for L, d, _ in rod_list])
    bs     = np.array([b for _, _, b in rod_list])[:, None]
    theta_true_avg = (bs * th_per).sum(0) / bs.sum()
    span_true      = float(theta_true_avg.max() - theta_true_avg.min())
    err_vs_true    = span_rec - span_true
    err_vs_60      = span_rec - true_span_deg
    results.append(dict(label=label, rod_list=rod_list,
                        theta_rec=theta_rec, theta_true=theta_true_avg,
                        span=span_rec, span_true=span_true,
                        err_vs_true=err_vs_true, err_vs_60=err_vs_60))

# ── Plot 1: theta_rec(psi) for every config, plus the true psi line ───────
n_z   = len(results)
n_col = 4
n_row = int(np.ceil(n_z / n_col))
if plt.fignum_exists("hinge_theta_rec"):
    plt.close("hinge_theta_rec")
fig_r, axes_r = plt.subplots(n_row, n_col, num="hinge_theta_rec",
                             figsize=(3.2 * n_col, 2.4 * n_row),
                             sharex=True, sharey=True, squeeze=False)
axes_r_flat = axes_r.ravel()

for ax, res in zip(axes_r_flat, results):
    ax.plot(psi_deg_hg, psi_deg_hg, "k:", lw=0.7, alpha=0.4,
            label=r"$\psi$ (arm sweep)")
    ax.plot(psi_deg_hg, res["theta_true"], "--", color="tab:blue", lw=1.0,
            label=r"true $\theta_\mathrm{lab}$ (bright. avg)")
    ax.plot(psi_deg_hg, res["theta_rec"], "-", color="tab:red", lw=1.4,
            label=r"$\theta_\mathrm{rec}$")
    ax.set_title(res["label"], fontsize=8)
    ax.text(0.04, 0.96,
            f"rec span={res['span']:.1f}°\n"
            f"true span={res['span_true']:.1f}°\n"
            f"Δvs true={res['err_vs_true']:+.1f}°",
            transform=ax.transAxes, fontsize=6, va="top", ha="left",
            bbox=dict(boxstyle="round,pad=0.2", fc="white",
                      ec="0.7", lw=0.5, alpha=0.9))
    ax.grid(alpha=0.3)
    ax.tick_params(labelsize=7)

for ax in axes_r_flat[len(results):]:
    ax.set_visible(False)

for ax in axes_r[-1, :]:
    ax.set_xlabel(r"$\psi$ (deg)", fontsize=8)
for ax in axes_r[:, 0]:
    ax.set_ylabel(r"$\theta$ (deg)", fontsize=8)

axes_r_flat[0].legend(fontsize=6, loc="lower right")
fig_r.suptitle(rf"Single-rod recovery of $\theta_\mathrm{{lab}}$ from "
               rf"combined ($a_x,a_y$)   true span = {true_span_deg:.0f}°",
               fontsize=11)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

# ── Plot 2: summary bar chart of recovered span error ─────────────────────
labels      = [r["label"]       for r in results]
spans       = np.array([r["span"]        for r in results])
spans_true  = np.array([r["span_true"]   for r in results])
err_true    = np.array([r["err_vs_true"] for r in results])

if plt.fignum_exists("hinge_theta_summary"):
    plt.close("hinge_theta_summary")
fig_s, (ax_s1, ax_s2) = plt.subplots(2, 1, num="hinge_theta_summary",
                                     figsize=(10, 6), sharex=True)

x = np.arange(len(labels))
w = 0.4
ax_s1.bar(x - w/2, spans_true, width=w, color="tab:blue",  label="true (per-rod, brightness-avg)")
ax_s1.bar(x + w/2, spans,      width=w, color="tab:red",   label="recovered (single-rod inversion)")
ax_s1.axhline(true_span_deg, color="k", ls=":", lw=0.8, label=f"ψ sweep = {true_span_deg:.0f}°")
ax_s1.set_ylabel(r"$\Delta\theta$ span (deg)")
ax_s1.legend(fontsize=8, loc="lower right")
ax_s1.grid(alpha=0.3, axis="y")

colors = ["tab:green" if abs(e) < 3 else
          "tab:orange" if abs(e) < 10 else "tab:red" for e in err_true]
ax_s2.bar(x, err_true, color=colors)
ax_s2.axhline(0, color="k", lw=0.8)
ax_s2.set_ylabel(r"recovered − true span (deg)")
ax_s2.set_xticks(x)
ax_s2.set_xticklabels(labels, rotation=35, ha="right", fontsize=7)
ax_s2.grid(alpha=0.3, axis="y")

fig_s.suptitle("Single-rod θ-recovery vs true brightness-weighted θ_lab span   "
               "(green: <3° error, orange: <10°, red: ≥10°)",
               fontsize=11)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

# ── Print table ───────────────────────────────────────────────────────────
print(f"{'config':<32} {'true span':>10} {'rec span':>10} {'Δ vs true':>10} {'Δ vs 60°':>10}")
print("-" * 78)
for r in results:
    print(f"{r['label']:<32} {r['span_true']:>10.2f} {r['span']:>10.2f} "
          f"{r['err_vs_true']:>+10.2f} {r['err_vs_60']:>+10.2f}")


In [ ]:
# ── Visual: why does Lambda < 90 reduce the theta_lab span? ───────────────
# Hinge axis k̂ in lab plane (Theta_axis_hg = 90, Phi_axis_hg = 0).
# For each Lambda, draw the rod m̂(psi) at psi = psi_min and psi = psi_max,
# and show the actual angle each rod makes with the lab z-axis.

from mpl_toolkits.mplot3d import Axes3D  # noqa: F401  (registers 3d projection)

k_hg, e1_hg, e2_hg = _axis_frame(Theta_axis_hg, Phi_axis_hg)

def _m(L_deg, psi_rad):
    L = np.radians(L_deg)
    return (np.cos(L) * k_hg
            + np.sin(L) * (np.cos(psi_rad) * e1_hg
                           + np.sin(psi_rad) * e2_hg))

Lambdas_show = [90.0, 75.0, 60.0]
psi_a = np.radians(psi_min_deg)   # 30°
psi_b = np.radians(psi_max_deg)   # 90°

if plt.fignum_exists("hinge_geom_explain"):
    plt.close("hinge_geom_explain")
fig_g = plt.figure(num="hinge_geom_explain", figsize=(4.6 * len(Lambdas_show), 4.4))

# Dense psi grid for the per-rod arc trace
psi_dense = np.linspace(0.0, 2 * np.pi, 360)

for i, L in enumerate(Lambdas_show, start=1):
    ax = fig_g.add_subplot(1, len(Lambdas_show), i, projection="3d")

    # Full circle traced by m̂(psi) for this Lambda (the cone of half-angle Λ
    # around k̂, intersected with the unit sphere).
    arc = np.array([_m(L, p) for p in psi_dense])
    ax.plot(arc[:, 0], arc[:, 1], arc[:, 2], "-", color="0.7", lw=0.8, alpha=0.7,
            label=rf"cone, $\Lambda$={L:.0f}°")

    # Hinge axis arrow
    ax.quiver(0, 0, 0, *k_hg, length=1.15, color="k", lw=2.0,
              arrow_length_ratio=0.10)
    ax.text(*(1.20 * k_hg), r"$\hat{\mathbf{k}}$", fontsize=11)

    # Lab z-axis (vertical reference)
    ax.quiver(0, 0, 0, 0, 0, 1, length=1.0, color="0.4", lw=1.0,
              arrow_length_ratio=0.10)
    ax.text(0.02, 0.02, 1.05, r"$\hat{\mathbf{z}}$", fontsize=10, color="0.4")

    for psi, c, lbl in [(psi_a, "tab:blue",   f"ψ={psi_min_deg:.0f}°"),
                       (psi_b, "tab:red",    f"ψ={psi_max_deg:.0f}°")]:
        m = _m(L, psi)
        th_lab = np.degrees(np.arccos(np.clip(m[2], -1, 1)))
        ax.quiver(0, 0, 0, *m, length=1.0, color=c, lw=2.4,
                  arrow_length_ratio=0.14, label=f"{lbl}  θ_lab={th_lab:.1f}°")
        ax.scatter([m[0]], [m[1]], [m[2]], s=55, c=c, edgecolors="k", lw=0.8,
                   zorder=6)

    # Compute and annotate the actual span Δθ_lab
    th_a = np.degrees(np.arccos(np.clip(_m(L, psi_a)[2], -1, 1)))
    th_b = np.degrees(np.arccos(np.clip(_m(L, psi_b)[2], -1, 1)))
    dth  = abs(th_b - th_a)

    ax.set_xlim(-1, 1); ax.set_ylim(-1, 1); ax.set_zlim(-1, 1)
    ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("z")
    ax.set_title(rf"$\Lambda$={L:.0f}°    "
                 rf"$\Delta\theta_\mathrm{{lab}}$ = {dth:.1f}°",
                 fontsize=10)
    ax.legend(fontsize=7, loc="upper left")
    ax.view_init(elev=20, azim=-65)

fig_g.suptitle("Rod orientation at ψ=30° and ψ=90° on a hinge axis along x̂  "
               "(k̂ = lab x).  Less tilt off the swept plane (Λ→90°) ⇒ full 60° span;  "
               "more tilt (Λ<90°) ⇒ rod stays farther from the substrate ⇒ smaller Δθ_lab.",
               fontsize=10)
plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.show()


In [ ]:
# ── Edge-on view: project the cone onto the y–z plane (look down k̂ = x̂) ──
# In this view the cone collapses to a circle of radius sin(Λ) centred at 0.
# The vertical coordinate of m̂(ψ) is m_z = sin(Λ)·cos(ψ),  and θ_lab = arccos(m_z).
# So θ_lab is read directly from the height of the arrow tip.
#
# Note on "θ_lab capping at 90°":
#   θ_lab is the angle between the rod m̂ and the lab z-axis (sample normal).
#   At ψ = 90° the rod lies in the x–y plane (the substrate), so m_z = 0 and
#   θ_lab = 90° — the rod is horizontal regardless of Λ.  That's not a cap from
#   the math; it's the geometry of "lying flat".  The other endpoint ψ = 30°
#   gives m_z = sin(Λ)·cos30° ≈ 0.866·sin(Λ), which shrinks with Λ — that's
#   why Δθ_lab < 60° for Λ < 90°.

Lambdas_show2 = [90.0, 75.0, 60.0]
psi_a = np.radians(psi_min_deg)   # 30°
psi_b = np.radians(psi_max_deg)   # 90°

if plt.fignum_exists("hinge_geom_yz"):
    plt.close("hinge_geom_yz")
fig_yz, axes_yz = plt.subplots(
    1, len(Lambdas_show2), figsize=(4.2 * len(Lambdas_show2), 4.4),
    num="hinge_geom_yz",
)

theta_circle = np.linspace(0.0, 2 * np.pi, 360)
unit_y = np.cos(theta_circle)
unit_z = np.sin(theta_circle)

for ax, L in zip(axes_yz, Lambdas_show2):
    sL = np.sin(np.radians(L))

    # Reference unit circle (radius 1)
    ax.plot(unit_y, unit_z, "--", color="0.75", lw=0.8, label="unit sphere ∩ y–z")
    # Cone projection: circle of radius sin(Λ)
    ax.plot(sL * unit_y, sL * unit_z, "-", color="0.4", lw=1.2,
            label=rf"cone projection (radius sin Λ = {sL:.3f})")

    # Axes
    ax.axhline(0, color="0.6", lw=0.6)
    ax.axvline(0, color="0.6", lw=0.6)
    # ẑ marker
    ax.annotate(r"$\hat{\mathbf{z}}$", xy=(0, 1), xytext=(0.04, 1.02),
                fontsize=11, color="0.3")
    # Substrate line (z = 0 = lab x–y plane edge-on)
    ax.axhspan(-1.15, 0, color="navajowhite", alpha=0.25, zorder=0)
    ax.text(-1.13, -0.08, "substrate (z<0)", fontsize=8, color="0.4")

    for psi, c, lbl in [(psi_a, "tab:blue", f"ψ={psi_min_deg:.0f}°"),
                        (psi_b, "tab:red",  f"ψ={psi_max_deg:.0f}°")]:
        # m̂ in lab frame, then project (drop x):
        my = -sL * np.sin(psi)
        mz =  sL * np.cos(psi)
        th_lab = np.degrees(np.arccos(np.clip(mz, -1, 1)))

        ax.annotate(
            "", xy=(my, mz), xytext=(0, 0),
            arrowprops=dict(arrowstyle="-|>", color=c, lw=2.4,
                            mutation_scale=18),
        )
        ax.plot([my], [mz], "o", color=c, ms=6, mec="k", mew=0.6)
        # Horizontal guide showing m_z
        ax.plot([0, my], [mz, mz], ":", color=c, lw=0.9, alpha=0.7)
        ax.text(-1.13, mz, f"m_z={mz:.3f}", fontsize=8, color=c, va="center")
        # Label at arrow tip
        ax.text(my * 1.08 - 0.02, mz * 1.08 + 0.03,
                f"{lbl}\nθ_lab={th_lab:.1f}°", fontsize=8, color=c,
                ha="right" if my < 0 else "left")

    th_a = np.degrees(np.arccos(np.clip(sL * np.cos(psi_a), -1, 1)))
    th_b = np.degrees(np.arccos(np.clip(sL * np.cos(psi_b), -1, 1)))
    dth = abs(th_b - th_a)

    ax.set_xlim(-1.2, 1.2)
    ax.set_ylim(-1.2, 1.2)
    ax.set_aspect("equal")
    ax.set_xlabel("y  (lab)")
    ax.set_ylabel("z  (lab)")
    ax.set_title(rf"$\Lambda$={L:.0f}°    "
                 rf"$\Delta\theta_\mathrm{{lab}}$ = {dth:.1f}°", fontsize=10)
    ax.legend(fontsize=7, loc="lower right")

fig_yz.suptitle(
    "Edge-on view (looking down k̂ = x̂):  m̂(ψ) traces a circle of radius sin(Λ) in the y–z plane.\n"
    "θ_lab = arccos(m_z) is read off the vertical coordinate of the arrow tip.",
    fontsize=10,
)
plt.tight_layout(rect=[0, 0, 1, 0.92])
plt.show()

In [ ]:
# ── ψ vs θ_lab: it's the hidden x-component, not the arc length ─────────
# Intuition trap: "ψ goes 60° around the cone circle, so θ_lab should go 60°."
# What's actually going on:
#   • In 3D, m̂ = (cos Λ, −sin Λ sin ψ, sin Λ cos ψ).  It's a UNIT vector,
#     but a fraction cos Λ of its length sits along x̂ (the hinge axis).
#   • cos θ_lab = m_z = sin Λ · cos ψ.  Only the z-component sets θ_lab.
#   • The y–z projection compresses m̂ to length sin Λ.  In that 2D picture
#     the angle from ẑ to the projected tip IS exactly ψ — but that 2D angle
#     is NOT θ_lab, because the missing m_x = cos Λ means the projected tip
#     is shorter than 1.  So at the same z-level the true 3D direction
#     (which is on the unit sphere) sits at a slightly larger angle from ẑ
#     than ψ does.
#
# Two equivalent ways to see why Δθ_lab < 60° for Λ < 90°:
#   (a) m_z range over ψ ∈ [30°, 90°] is [0, sin Λ · cos 30°].  For Λ=90°
#       the upper end is cos 30° = 0.866 ⇒ θ_lab ranges [30°, 90°] = 60°.
#       For Λ=75° it's 0.836 ⇒ θ_lab ranges [33.3°, 90°] = 56.7°.
#   (b) The rod can never get closer to ẑ than θ_min = 90° − Λ (the cone
#       intersects ẑ only when Λ = 90°).  As Λ shrinks, θ_lab(ψ=30°) is
#       lifted above 30°, while θ_lab(ψ=90°) stays pinned at 90° (lying flat).
#
# The figure below makes this visual: same z-level on the small cone circle
# (radius sin Λ) projects horizontally to a different angle on the unit
# circle.  Purple arc = Δψ on cone circle.  Green arc = Δθ_lab on unit circle.

from matplotlib.patches import Arc

Lambdas_demo = [90.0, 75.0, 60.0]
psi_a = np.radians(psi_min_deg)
psi_b = np.radians(psi_max_deg)

if plt.fignum_exists("hinge_geom_arcs"):
    plt.close("hinge_geom_arcs")
fig_a, axes_a = plt.subplots(
    1, len(Lambdas_demo), figsize=(4.6 * len(Lambdas_demo), 4.8),
    num="hinge_geom_arcs",
)

theta_circle = np.linspace(0.0, 2 * np.pi, 360)
unit_y = np.cos(theta_circle)
unit_z = np.sin(theta_circle)

for ax, L in zip(axes_a, Lambdas_demo):
    sL = np.sin(np.radians(L))

    # Unit circle
    ax.plot(unit_y, unit_z, "--", color="0.7", lw=0.8, label="unit circle")
    # Cone projection circle (radius sin Λ, centred at origin in this projection)
    ax.plot(sL * unit_y, sL * unit_z, "-", color="0.45", lw=1.3,
            label=rf"cone circle, r=sin Λ={sL:.3f}")

    # Reference axes
    ax.axhline(0, color="0.7", lw=0.6)
    ax.axvline(0, color="0.7", lw=0.6)
    ax.annotate(r"$\hat{\mathbf{z}}$", xy=(0, 1.02), fontsize=12, color="0.3")

    # m̂ tips at ψ = 30° and ψ = 90° (on the small circle)
    pts = {}
    for psi, c, name in [(psi_a, "tab:blue", "ψ=30°"),
                         (psi_b, "tab:red",  "ψ=90°")]:
        my = -sL * np.sin(psi)
        mz =  sL * np.cos(psi)
        pts[name] = (my, mz, c)
        ax.annotate("", xy=(my, mz), xytext=(0, 0),
                    arrowprops=dict(arrowstyle="-|>", color=c, lw=2.4,
                                    mutation_scale=18))
        ax.plot([my], [mz], "o", color=c, ms=6, mec="k", mew=0.6)

        # Horizontal "m_z = const" line out to the unit circle.
        # On the unit circle at the same z: y_uc = −sqrt(1 − m_z²).
        y_uc = -np.sqrt(max(0.0, 1.0 - mz**2))
        ax.plot([my, y_uc], [mz, mz], ":", color=c, lw=1.0, alpha=0.8)
        ax.plot([y_uc], [mz], "s", color=c, ms=6, mec="k", mew=0.6)
        # Faint dotted line from origin to that unit-circle point
        # (this is the direction whose angle from +z IS θ_lab):
        ax.plot([0, y_uc], [0, mz], "-", color=c, lw=0.8, alpha=0.35)

    # ── ψ arc: 60° on the cone circle, vertex at origin (= cone centre here) ──
    # ψ measured from ẑ axis going toward −y (since m_y = −sin Λ sin ψ).
    # Draw it at 80% radius so it sits inside the cone circle.
    r_psi = 0.80 * sL
    ax.add_patch(Arc((0, 0), 2 * r_psi, 2 * r_psi, angle=0,
                     theta1=90 + np.degrees(-psi_b),     # reaches ψ=90° (lying flat, −y dir)
                     theta2=90 + np.degrees(-psi_a),     # starts at ψ=30°
                     color="purple", lw=2.4))
    # ψ-arc label
    mid_psi = 0.5 * (psi_a + psi_b)
    lpx, lpz = -r_psi * np.sin(mid_psi) * 0.85, r_psi * np.cos(mid_psi) * 0.85
    ax.text(lpx - 0.05, lpz, "Δψ = 60°", color="purple", fontsize=9,
            ha="right", va="center", fontweight="bold")

    # ── θ_lab arc: from ẑ to m̂ at origin, on the unit circle ──
    # θ_lab(ψ) = arccos(sin Λ · cos ψ).  Draw at radius 1.05.
    th_a = np.arccos(np.clip(sL * np.cos(psi_a), -1, 1))
    th_b = np.arccos(np.clip(sL * np.cos(psi_b), -1, 1))
    r_th = 1.06
    ax.add_patch(Arc((0, 0), 2 * r_th, 2 * r_th, angle=0,
                     theta1=90 - np.degrees(th_b),
                     theta2=90 - np.degrees(th_a),
                     color="darkgreen", lw=2.4))
    mid_th = 0.5 * (th_a + th_b)
    ltx, ltz = -r_th * np.sin(mid_th) * 1.05, r_th * np.cos(mid_th) * 1.05
    ax.text(ltx - 0.02, ltz, f"Δθ_lab = {np.degrees(th_b - th_a):.1f}°",
            color="darkgreen", fontsize=9, ha="right", va="center",
            fontweight="bold")

    # Per-arrow θ_lab labels near tips
    for name, (my, mz, c) in pts.items():
        th = np.degrees(np.arccos(np.clip(mz, -1, 1)))
        ax.text(my * 1.10 - 0.02, mz * 1.10 + 0.04,
                f"{name}\nθ_lab={th:.1f}°", fontsize=8, color=c,
                ha="right" if my < 0 else "left")

    ax.set_xlim(-1.35, 1.35)
    ax.set_ylim(-1.25, 1.35)
    ax.set_aspect("equal")
    ax.set_xlabel("y  (lab)")
    ax.set_ylabel("z  (lab)")
    ax.set_title(rf"$\Lambda$={L:.0f}°", fontsize=11)
    ax.legend(fontsize=7, loc="lower right")

fig_a.suptitle(
    "Solid arrow = rod tip on the cone circle (radius sin Λ).  Square = same z-level on the unit circle.\n"
    "Δψ (purple) = 60° around the cone circle.   Δθ_lab (green) = arc on the unit circle at the matching z-levels.\n"
    "θ_lab depends only on m_z = sin Λ · cos ψ, so when the cone circle is shorter (Λ<90°) the z-range shrinks and Δθ_lab < 60°.",
    fontsize=9.5,
)
plt.tight_layout(rect=[0, 0, 1, 0.90])
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

# ── Two-arm sheet structure ──────────────────────────────────────────────
# Each arm is a flat rectangular sheet:
#   length = 10  (along the arm)
#   width  =  1  (perpendicular to the arm, in the sheet plane)
#   height =  0  (zero thickness)
#
# Arm 1: lies flat on the ground (xy-plane), extending along +y.
# Arm 2: shares the hinge edge with arm 1 at the origin, rotated about
#        the x-axis so it makes a 60° angle with arm 1 (and the ground).

ARM_LEN   = 10.0
ARM_WIDTH = 1.0
HINGE_DEG = 60.0

def make_sheet(length, width, hinge_deg):
    """Rectangle in the y-axis-along plane, hinged at x-axis, rotated by hinge_deg about x.

    Corners ordered as a quad (CCW when viewed from +normal).
    Width is centered on x = 0 so the hinge edge is the segment x ∈ [-w/2, w/2], y=0, z=0.
    """
    h = np.radians(hinge_deg)
    # Local corners in the arm's own plane (before rotation): y ∈ [0, L], x ∈ [-w/2, w/2], z = 0.
    local = np.array([
        [-width / 2, 0.0,    0.0],
        [+width / 2, 0.0,    0.0],
        [+width / 2, length, 0.0],
        [-width / 2, length, 0.0],
    ])
    # Rotation about x-axis by hinge_deg lifts the +y edge up into +z.
    R = np.array([
        [1.0,         0.0,          0.0],
        [0.0, np.cos(h), -np.sin(h)],
        [0.0, np.sin(h),  np.cos(h)],
    ])
    return local @ R.T

arm1 = make_sheet(ARM_LEN, ARM_WIDTH, hinge_deg=0.0)
arm2 = make_sheet(ARM_LEN, ARM_WIDTH, hinge_deg=HINGE_DEG)

# ── Plot ─────────────────────────────────────────────────────────────────
if plt.fignum_exists("two_arm_sheets"):
    plt.close("two_arm_sheets")
fig = plt.figure(num="two_arm_sheets", figsize=(9, 7))
ax = fig.add_subplot(111, projection="3d")

# Faint ground plane for reference
gx, gy = np.meshgrid(np.linspace(-2, 2, 2), np.linspace(-1, ARM_LEN + 1, 2))
ax.plot_surface(gx, gy, np.zeros_like(gx),
                color="navajowhite", alpha=0.20, edgecolor="none")

# Arm sheets
poly1 = Poly3DCollection([arm1], facecolor="tab:gray",   edgecolor="k",
                         linewidths=1.2, alpha=0.65)
poly2 = Poly3DCollection([arm2], facecolor="tab:purple", edgecolor="k",
                         linewidths=1.2, alpha=0.65)
ax.add_collection3d(poly1)
ax.add_collection3d(poly2)

# Hinge edge (shared between the two arms) drawn explicitly
ax.plot([-ARM_WIDTH / 2, ARM_WIDTH / 2], [0, 0], [0, 0],
        color="k", lw=2.5, label="hinge edge")

# Arm centerlines for reference
ax.plot([0, 0], [0, ARM_LEN], [0, 0],
        color="tab:gray", lw=2, label="arm 1 (on ground)")
end2 = ARM_LEN * np.array([0.0,
                           np.cos(np.radians(HINGE_DEG)),
                           np.sin(np.radians(HINGE_DEG))])
ax.plot([0, end2[0]], [0, end2[1]], [0, end2[2]],
        color="tab:purple", lw=2, label=f"arm 2 ({HINGE_DEG:.0f}° to arm 1)")

# Hinge angle arc in the y-z plane at the origin
arc_t = np.linspace(0, np.radians(HINGE_DEG), 60)
r_arc = 1.5
ax.plot(np.zeros_like(arc_t),
        r_arc * np.cos(arc_t),
        r_arc * np.sin(arc_t),
        color="0.3", lw=1.2)
ax.text(0,
        r_arc * 0.95 * np.cos(np.radians(HINGE_DEG / 2)),
        r_arc * 0.95 * np.sin(np.radians(HINGE_DEG / 2)) + 0.2,
        f"{HINGE_DEG:.0f}°", color="0.2", fontsize=11)

ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("z (up)")
ax.set_xlim(-2, 2)
ax.set_ylim(-1, ARM_LEN + 1)
ax.set_zlim(0, ARM_LEN)
try:
    ax.set_box_aspect((4, ARM_LEN + 2, ARM_LEN))
except Exception:
    pass
ax.set_title(
    f"Two-arm sheet structure  (each arm: {ARM_LEN:.0f} × {ARM_WIDTH:.0f}, zero thickness)\n"
    f"Arm 1 flat on ground; arm 2 hinged at {HINGE_DEG:.0f}° above arm 1.",
    fontsize=10,
)
ax.legend(loc="upper left", fontsize=8)
ax.view_init(elev=22, azim=-60)
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

# ── Add a rod attached to the upper arm ──────────────────────────────────
# Geometry (re-stated, self-contained):
#   Arm 1 (lower): 10 × 1, flat on the ground, centered on x = 0, along +y.
#   Arm 2 (upper): 10 × 1, hinged at the x-axis, tilted HINGE_DEG above ground.
#
# Arm-local frame (right-handed) attached to arm 2:
#   x_arm  = along the arm's LONG axis (length direction)
#   y_arm  = along the arm's WIDTH axis (in the sheet plane, ⊥ length)
#   z_arm  = normal to the sheet (out-of-plane)
#
# Rod:
#   length = 1
#   center sits 2 units below the TOP of the arm, on the width centerline,
#       i.e. at (x_arm, y_arm, z_arm) = (ARM_LEN - 2, 0, 0)
#   orientation in the arm-local frame is given by spherical (θ, φ):
#       rod_hat_local = (sin θ cos φ, sin θ sin φ, cos θ)
#   so θ is measured from z_arm (sheet normal) and φ is measured from
#   x_arm (long axis) toward y_arm (width axis).
#   The rod is allowed to pass through the sheet.

ARM_LEN   = 10.0
ARM_WIDTH = 1.0
HINGE_DEG = 60.0
ROD_LEN   = 1.0
ROD_OFFSET_FROM_TOP = 2.0   # rod center is this far below the top of arm 2

# Arm-2 local basis expressed in the LAB frame.
h = np.radians(HINGE_DEG)
e_x_arm = np.array([0.0,  np.cos(h),  np.sin(h)])   # along length (long axis)
e_y_arm = np.array([-1.0, 0.0,        0.0])         # along width (right-handed)
e_z_arm = np.array([0.0, -np.sin(h),  np.cos(h)])   # sheet normal

assert np.allclose(np.cross(e_x_arm, e_y_arm), e_z_arm)

def rod_local_to_lab(theta, phi):
    """Unit rod direction in lab frame, given arm-local spherical angles."""
    rod_local = np.array([
        np.sin(theta) * np.cos(phi),
        np.sin(theta) * np.sin(phi),
        np.cos(theta),
    ])
    return (rod_local[0] * e_x_arm
            + rod_local[1] * e_y_arm
            + rod_local[2] * e_z_arm)

# ── Generate a random rod orientation (uniform on the sphere) ────────────
rng   = np.random.default_rng(seed=7)
theta = np.arccos(rng.uniform(-1.0, 1.0))
phi   = rng.uniform(0.0, 2.0 * np.pi)

# Rod center and endpoints in lab frame
rod_center_local = np.array([ARM_LEN - ROD_OFFSET_FROM_TOP, 0.0, 0.0])
rod_center_lab = (rod_center_local[0] * e_x_arm
                  + rod_center_local[1] * e_y_arm
                  + rod_center_local[2] * e_z_arm)

rod_dir_lab = rod_local_to_lab(theta, phi)
rod_p1 = rod_center_lab - 0.5 * ROD_LEN * rod_dir_lab
rod_p2 = rod_center_lab + 0.5 * ROD_LEN * rod_dir_lab

# Lab-frame spherical angles of the rod direction
# θ_lab from +z (lab vertical), φ_lab from +x toward +y, in [0, 2π).
theta_lab = np.arccos(np.clip(rod_dir_lab[2], -1.0, 1.0))
phi_lab   = np.arctan2(rod_dir_lab[1], rod_dir_lab[0]) % (2.0 * np.pi)

print("Random rod orientation")
print(f"  arm-local:   θ = {np.degrees(theta):7.3f}°   "
      f"φ = {np.degrees(phi):7.3f}°")
print(f"  lab frame:   θ_lab = {np.degrees(theta_lab):7.3f}° (from +z)   "
      f"φ_lab = {np.degrees(phi_lab):7.3f}° (from +x)")

# ── Plot ─────────────────────────────────────────────────────────────────
def make_sheet(length, width, hinge_deg):
    h = np.radians(hinge_deg)
    local = np.array([
        [-width / 2, 0.0,    0.0],
        [+width / 2, 0.0,    0.0],
        [+width / 2, length, 0.0],
        [-width / 2, length, 0.0],
    ])
    R = np.array([
        [1.0,         0.0,          0.0],
        [0.0, np.cos(h), -np.sin(h)],
        [0.0, np.sin(h),  np.cos(h)],
    ])
    return local @ R.T

arm1 = make_sheet(ARM_LEN, ARM_WIDTH, hinge_deg=0.0)
arm2 = make_sheet(ARM_LEN, ARM_WIDTH, hinge_deg=HINGE_DEG)

if plt.fignum_exists("two_arm_with_rod"):
    plt.close("two_arm_with_rod")
fig = plt.figure(num="two_arm_with_rod", figsize=(9, 7))
ax = fig.add_subplot(111, projection="3d")

# Ground
gx, gy = np.meshgrid(np.linspace(-2, 2, 2), np.linspace(-1, ARM_LEN + 1, 2))
ax.plot_surface(gx, gy, np.zeros_like(gx),
                color="navajowhite", alpha=0.20, edgecolor="none")

# Arm sheets
ax.add_collection3d(Poly3DCollection([arm1], facecolor="tab:gray",
                                     edgecolor="k", linewidths=1.2, alpha=0.55))
ax.add_collection3d(Poly3DCollection([arm2], facecolor="tab:purple",
                                     edgecolor="k", linewidths=1.2, alpha=0.45))

# Hinge edge
ax.plot([-ARM_WIDTH / 2, ARM_WIDTH / 2], [0, 0], [0, 0],
        color="k", lw=2.0, label="hinge edge")

# Arm-local basis at the rod-center (small reference arrows)
basis_len = 0.9
for vec, color, name in [(e_x_arm, "tab:red",   "x_arm (length)"),
                         (e_y_arm, "tab:green", "y_arm (width)"),
                         (e_z_arm, "tab:blue",  "z_arm (normal)")]:
    end = rod_center_lab + basis_len * vec
    ax.plot([rod_center_lab[0], end[0]],
            [rod_center_lab[1], end[1]],
            [rod_center_lab[2], end[2]],
            color=color, lw=1.6)
    ax.text(end[0], end[1], end[2], "  " + name, color=color, fontsize=8)

# Lab-frame basis at the origin (dashed) for reference
lab_origin = np.array([0.0, 0.0, 0.0])
lab_len = 1.2
for vec, name in [(np.array([1.0, 0.0, 0.0]), "x_lab"),
                  (np.array([0.0, 1.0, 0.0]), "y_lab"),
                  (np.array([0.0, 0.0, 1.0]), "z_lab")]:
    end = lab_origin + lab_len * vec
    ax.plot([0, end[0]], [0, end[1]], [0, end[2]],
            color="0.3", lw=1.0, ls="--")
    ax.text(end[0], end[1], end[2], "  " + name, color="0.3", fontsize=7)

# Rod
ax.plot([rod_p1[0], rod_p2[0]],
        [rod_p1[1], rod_p2[1]],
        [rod_p1[2], rod_p2[2]],
        color="tab:orange", lw=4,
        label=(f"rod  arm-local (θ={np.degrees(theta):.1f}°, "
               f"φ={np.degrees(phi):.1f}°)\n"
               f"      lab       (θ_lab={np.degrees(theta_lab):.1f}°, "
               f"φ_lab={np.degrees(phi_lab):.1f}°)"))
ax.scatter(*rod_center_lab, s=40, color="tab:orange", edgecolors="k",
           lw=0.6, zorder=6, label="rod center")

ax.set_xlabel("x (lab)"); ax.set_ylabel("y (lab)"); ax.set_zlabel("z (lab, up)")
ax.set_xlim(-2, 2)
ax.set_ylim(-1, ARM_LEN + 1)
ax.set_zlim(0, ARM_LEN)
try:
    ax.set_box_aspect((4, ARM_LEN + 2, ARM_LEN))
except Exception:
    pass
ax.set_title(
    f"Random rod on upper arm  (rod length={ROD_LEN}, center {ROD_OFFSET_FROM_TOP} below top)\n"
    f"arm-local: θ={np.degrees(theta):.1f}°, φ={np.degrees(phi):.1f}°   |   "
    f"lab: θ_lab={np.degrees(theta_lab):.1f}°, φ_lab={np.degrees(phi_lab):.1f}°",
    fontsize=10,
)
ax.legend(loc="upper left", fontsize=8)
ax.view_init(elev=22, azim=-60)
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ── Sweep the hinge angle, keep rod-vs-arm orientation FIXED ────────────
# Rod is fixed in the arm-local frame at (theta_rod_local, phi_rod_local).
# We sweep the hinge angle of arm 2 from 0° to 60° and, for each hinge:
#   1) build the arm-local basis in the lab frame,
#   2) rotate the (fixed) rod direction into the lab frame,
#   3) read off lab θ_lab and φ_lab,
#   4) compute Fourkas anisotropy r and (a_x, a_y) = r·(cos 2φ, sin 2φ).

# Pick a fixed rod-vs-arm orientation. Keep θ_rod_local well away from 0/π so
# φ_rod_local is meaningful.
theta_rod_local = np.radians(90.0)   # tilt away from sheet normal
phi_rod_local   = np.radians(0.0)   # azimuth in arm plane

# Reuse Fourkas A,B,C from earlier cells if available; otherwise compute with
# defaults consistent with the earlier definition.
try:
    A, B, C
    _ABC_source = "from earlier cell"
except NameError:
    NA = 1.3
    nw = 1.33
    alpha = np.arcsin(NA / nw)
    ca = np.cos(alpha)
    A = 1/6 - ca/4 + ca**3/12
    B = ca/8 - ca**3/8
    C = 7/48 - ca/16 - ca**2/16 - ca**3/48
    _ABC_source = "computed locally"
print(f"Fourkas coefficients ({_ABC_source}): A={A:.5f}  B={B:.5f}  C={C:.5f}")

def arm_basis(hinge_deg):
    h = np.radians(hinge_deg)
    e_x = np.array([0.0,  np.cos(h),  np.sin(h)])
    e_y = np.array([-1.0, 0.0,        0.0])
    e_z = np.array([0.0, -np.sin(h),  np.cos(h)])
    return e_x, e_y, e_z

hinge_sweep_deg = np.linspace(0.0, 60.0, 121)
theta_lab_sweep = np.empty_like(hinge_sweep_deg)
phi_lab_sweep   = np.empty_like(hinge_sweep_deg)

# Rod direction in arm-local coordinates (fixed throughout sweep)
rod_local = np.array([
    np.sin(theta_rod_local) * np.cos(phi_rod_local),
    np.sin(theta_rod_local) * np.sin(phi_rod_local),
    np.cos(theta_rod_local),
])

for i, hd in enumerate(hinge_sweep_deg):
    ex, ey, ez = arm_basis(hd)
    rod_lab = rod_local[0] * ex + rod_local[1] * ey + rod_local[2] * ez
    theta_lab_sweep[i] = np.arccos(np.clip(rod_lab[2], -1.0, 1.0))
    phi_lab_sweep[i]   = np.arctan2(rod_lab[1], rod_lab[0])

# Fourkas anisotropy
s2 = np.sin(theta_lab_sweep) ** 2
r_sweep  = C * s2 / (A + B * s2)
ax_sweep = r_sweep * np.cos(2.0 * phi_lab_sweep)
ay_sweep = r_sweep * np.sin(2.0 * phi_lab_sweep)
R_SAT_local = C / (A + B)

# ── Plot ─────────────────────────────────────────────────────────────────
if plt.fignum_exists("hinge_sweep_aniso"):
    plt.close("hinge_sweep_aniso")
fig, axes = plt.subplots(2, 2, figsize=(11, 8), num="hinge_sweep_aniso")

# (a) θ_lab vs hinge
ax_t = axes[0, 0]
ax_t.plot(hinge_sweep_deg, np.degrees(theta_lab_sweep), color="tab:blue", lw=2)
ax_t.set_xlabel("hinge angle (deg)")
ax_t.set_ylabel(r"$\theta_{\rm lab}$ (deg, from +z)")
ax_t.set_title(r"$\theta_{\rm lab}$ vs hinge angle")
ax_t.grid(alpha=0.25)

# (b) φ_lab vs hinge
ax_p = axes[0, 1]
ax_p.plot(hinge_sweep_deg,
          (np.degrees(phi_lab_sweep) % 360.0),
          color="tab:green", lw=2)
ax_p.set_xlabel("hinge angle (deg)")
ax_p.set_ylabel(r"$\phi_{\rm lab}$ (deg, from +x, in [0,360))")
ax_p.set_title(r"$\phi_{\rm lab}$ vs hinge angle")
ax_p.grid(alpha=0.25)

# (c) anisotropy r vs hinge
ax_r = axes[1, 0]
ax_r.plot(hinge_sweep_deg, r_sweep, color="tab:red", lw=2, label="r(hinge)")
ax_r.axhline(R_SAT_local, color="0.5", ls="--", lw=1,
             label=f"R_SAT = {R_SAT_local:.3f}")
ax_r.set_xlabel("hinge angle (deg)")
ax_r.set_ylabel("anisotropy radius  r")
ax_r.set_title(r"$r(\theta_{\rm lab})$ vs hinge angle")
ax_r.grid(alpha=0.25)
ax_r.legend(fontsize=8)

# (d) (a_x, a_y) trajectory, coloured by hinge
ax_xy = axes[1, 1]
sc = ax_xy.scatter(ax_sweep, ay_sweep, c=hinge_sweep_deg,
                   cmap="viridis", s=14)
ax_xy.plot(ax_sweep, ay_sweep, "-", color="0.7", lw=0.8, alpha=0.8)
# Reference saturation circle
tt = np.linspace(0, 2 * np.pi, 360)
ax_xy.plot(R_SAT_local * np.cos(tt), R_SAT_local * np.sin(tt),
           ":", color="0.6", lw=1, label=f"|a|=R_SAT={R_SAT_local:.3f}")
ax_xy.scatter([ax_sweep[0]], [ay_sweep[0]], s=70, marker="o",
              facecolor="white", edgecolor="k", zorder=5,
              label=f"hinge={hinge_sweep_deg[0]:.0f}°")
ax_xy.scatter([ax_sweep[-1]], [ay_sweep[-1]], s=70, marker="s",
              facecolor="white", edgecolor="k", zorder=5,
              label=f"hinge={hinge_sweep_deg[-1]:.0f}°")
ax_xy.set_aspect("equal")
ax_xy.set_xlabel(r"$a_x = r\cos 2\phi_{\rm lab}$")
ax_xy.set_ylabel(r"$a_y = r\sin 2\phi_{\rm lab}$")
ax_xy.set_title("Anisotropy trajectory while hinge sweeps 0°→60°")
ax_xy.grid(alpha=0.25)
ax_xy.legend(fontsize=8, loc="upper right")
plt.colorbar(sc, ax=ax_xy, label="hinge angle (deg)", shrink=0.85)

fig.suptitle(
    f"Rod fixed in arm frame at (θ_rod={np.degrees(theta_rod_local):.0f}°, "
    f"φ_rod={np.degrees(phi_rod_local):.0f}°);  "
    f"hinge of arm 2 swept 0°→60°.\n"
    r"Anisotropy via Fourkas:  $r=\frac{C\sin^2\theta_{\rm lab}}{A+B\sin^2\theta_{\rm lab}}$,"
    r"  $(a_x,a_y)=r(\cos 2\phi_{\rm lab},\sin 2\phi_{\rm lab})$.",
    fontsize=10,
)
plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

# ── Multi-rod ensemble on upper arm: anisotropy vs hinge angle ──────────
# Each rod is fixed in the arm-local frame:
#   u         : position along the arm length, in [0, ARM_LEN]
#   v         : position along the arm width,  in [-ARM_WIDTH/2, ARM_WIDTH/2]
#   theta_deg : rod tilt from the sheet normal (z_arm), in [0, 180]
#   phi_deg   : rod azimuth from the long axis (x_arm) toward width (y_arm),
#               in [0, 360)
#   b         : brightness weight (>= 0)
#
# Edit `rods` below to add/remove/configure rods manually.

ARM_LEN   = 10.0
ARM_WIDTH = 1.0

# ──────────────────────────── MANUAL CONFIG ─────────────────────────────
# Add/remove dicts below. Each dict is one rod.
rods = [
    dict(u=8.0, v=+0.30, theta_deg=90.0, phi_deg= 0.0, b=1.0),
    dict(u=4.0, v=-0.20, theta_deg=75.0, phi_deg=150.0, b=0.8),
    dict(u=2.0, v= 0.00, theta_deg=40.0, phi_deg=300.0, b=1.2),
]

# Hinge sweep range
HINGE_MIN_DEG = 0.0
HINGE_MAX_DEG = 60.0
N_HINGE       = 121
# ────────────────────────────────────────────────────────────────────────

n_rods = len(rods)

# Reuse Fourkas A,B,C from earlier cells.
try:
    _A, _B, _C = A_a, B_a, C_a
except NameError:
    _A, _B, _C = A, B, C
print(f"Fourkas (A, B, C) = ({_A:.5f}, {_B:.5f}, {_C:.5f})")
print(f"n_rods = {n_rods}")
for i, r in enumerate(rods):
    print(f"  rod {i}: u={r['u']:.2f}, v={r['v']:+.2f}, "
          f"θ={r['theta_deg']:.1f}°, φ={r['phi_deg']:.1f}°, b={r['b']:.2f}")

def _r_to_theta(r):
    R_SAT = _C / (_A + _B)
    rc = np.clip(r, 0.0, R_SAT)
    s2 = rc * _A / (_C - rc * _B)
    s2 = np.clip(s2, 0.0, 1.0)
    return np.arcsin(np.sqrt(s2))

def rod_dir_local(theta_deg, phi_deg):
    th = np.radians(theta_deg)
    ph = np.radians(phi_deg)
    return np.array([np.sin(th) * np.cos(ph),
                     np.sin(th) * np.sin(ph),
                     np.cos(th)])

def arm_basis(hinge_deg):
    h = np.radians(hinge_deg)
    e_x = np.array([0.0,  np.cos(h),  np.sin(h)])    # length
    e_y = np.array([-1.0, 0.0,        0.0])          # width
    e_z = np.array([0.0, -np.sin(h),  np.cos(h)])    # sheet normal
    return e_x, e_y, e_z

# ── Sweep hinge: per-rod truth + intensity-sum anisotropy ───────────────
hinge_deg_arr = np.linspace(HINGE_MIN_DEG, HINGE_MAX_DEG, N_HINGE)
N_h = len(hinge_deg_arr)

theta_lab_per = np.zeros((n_rods, N_h))
phi_lab_per   = np.zeros((n_rods, N_h))
I0   = np.zeros(N_h)
I45  = np.zeros(N_h)
I90  = np.zeros(N_h)
I135 = np.zeros(N_h)

for j, hd in enumerate(hinge_deg_arr):
    ex, ey, ez = arm_basis(hd)
    for i, r in enumerate(rods):
        m_loc = rod_dir_local(r["theta_deg"], r["phi_deg"])
        m_lab = m_loc[0] * ex + m_loc[1] * ey + m_loc[2] * ez
        th = np.arccos(np.clip(m_lab[2], -1.0, 1.0))
        ph = np.arctan2(m_lab[1], m_lab[0])
        theta_lab_per[i, j] = th
        phi_lab_per[i, j]   = ph
        s2   = np.sin(th) ** 2
        base = _A + _B * s2
        cos2 = _C * s2 * np.cos(2.0 * ph)
        sin2 = _C * s2 * np.sin(2.0 * ph)
        b    = r["b"]
        I0[j]   += b * (base + cos2)
        I45[j]  += b * (base + sin2)
        I90[j]  += b * (base - cos2)
        I135[j] += b * (base - sin2)

rx = (I0  - I90 ) / (I0  + I90 )
ry = (I45 - I135) / (I45 + I135)

# Per-rod anisotropy (each rod treated alone)
s2_per = np.sin(theta_lab_per) ** 2
r_per  = (_C * s2_per) / (_A + _B * s2_per)
rx_per = r_per * np.cos(2.0 * phi_lab_per)
ry_per = r_per * np.sin(2.0 * phi_lab_per)

# Single-rod recovery from (rx, ry)
r_amp     = np.sqrt(rx ** 2 + ry ** 2)
theta_rec = _r_to_theta(r_amp)
phi_rec   = 0.5 * np.arctan2(ry, rx)   # ∈ (-π/2, π/2]

# ── Plot ────────────────────────────────────────────────────────────────
def make_sheet(length, width, hinge_deg):
    h = np.radians(hinge_deg)
    local = np.array([
        [-width / 2, 0.0,    0.0],
        [+width / 2, 0.0,    0.0],
        [+width / 2, length, 0.0],
        [-width / 2, length, 0.0],
    ])
    R = np.array([
        [1.0,         0.0,          0.0],
        [0.0, np.cos(h), -np.sin(h)],
        [0.0, np.sin(h),  np.cos(h)],
    ])
    return local @ R.T

if plt.fignum_exists("multi_rod_hinge"):
    plt.close("multi_rod_hinge")
fig = plt.figure(num="multi_rod_hinge", figsize=(13, 9))

# (1) 3D structure with the rods at hinge=HINGE_MAX
ax3 = fig.add_subplot(2, 3, 1, projection="3d")
H_show = HINGE_MAX_DEG
arm1_v = make_sheet(ARM_LEN, ARM_WIDTH, hinge_deg=0.0)
arm2_v = make_sheet(ARM_LEN, ARM_WIDTH, hinge_deg=H_show)
ax3.add_collection3d(Poly3DCollection([arm1_v], facecolor="tab:gray",
                                       edgecolor="k", linewidths=1.0, alpha=0.45))
ax3.add_collection3d(Poly3DCollection([arm2_v], facecolor="tab:purple",
                                       edgecolor="k", linewidths=1.0, alpha=0.40))

ROD_DRAW_LEN = 1.0
ex_v, ey_v, ez_v = arm_basis(H_show)
colors = ["tab:orange", "tab:cyan", "tab:olive", "tab:pink", "tab:brown"]
for i, r in enumerate(rods):
    center_lab = r["u"] * ex_v + r["v"] * ey_v
    m_loc = rod_dir_local(r["theta_deg"], r["phi_deg"])
    m_lab = m_loc[0] * ex_v + m_loc[1] * ey_v + m_loc[2] * ez_v
    p1 = center_lab - 0.5 * ROD_DRAW_LEN * m_lab
    p2 = center_lab + 0.5 * ROD_DRAW_LEN * m_lab
    ax3.plot([p1[0], p2[0]], [p1[1], p2[1]], [p1[2], p2[2]],
             color=colors[i % len(colors)], lw=3,
             label=f"rod {i}: θ={r['theta_deg']:.0f}°, φ={r['phi_deg']:.0f}°, b={r['b']:.2f}")
    ax3.scatter(*center_lab, s=25, color=colors[i % len(colors)],
                edgecolors="k", lw=0.5, zorder=6)
ax3.set_xlabel("x"); ax3.set_ylabel("y"); ax3.set_zlabel("z")
ax3.set_xlim(-2, 2); ax3.set_ylim(-1, ARM_LEN + 1); ax3.set_zlim(0, ARM_LEN)
try:
    ax3.set_box_aspect((4, ARM_LEN + 2, ARM_LEN))
except Exception:
    pass
ax3.set_title(f"{n_rods} rod(s) on upper arm  (shown at hinge={H_show:.0f}°)",
              fontsize=10)
ax3.legend(loc="upper left", fontsize=7)
ax3.view_init(elev=22, azim=-60)

# (2) rx, ry vs hinge
ax_rx = fig.add_subplot(2, 3, 2)
ax_rx.plot(hinge_deg_arr, rx, color="tab:blue",  lw=2, label=r"$r_x$")
ax_rx.plot(hinge_deg_arr, ry, color="tab:green", lw=2, label=r"$r_y$")
ax_rx.set_xlabel("hinge angle (deg)")
ax_rx.set_ylabel("anisotropy")
ax_rx.set_title("Intensity-sum anisotropy")
ax_rx.grid(alpha=0.25); ax_rx.legend(fontsize=8)

# (3) (rx, ry) trajectory: per-rod individual + intensity-sum combined
ax_xy = fig.add_subplot(2, 3, 3)
# Per-rod individual trajectories
for i in range(n_rods):
    ax_xy.plot(rx_per[i], ry_per[i], "-",
               color=colors[i % len(colors)], lw=1.2, alpha=0.7,
               label=f"rod {i}")
    ax_xy.scatter([rx_per[i, 0]],  [ry_per[i, 0]],  s=30, marker="o",
                  facecolor=colors[i % len(colors)], edgecolor="k",
                  lw=0.5, zorder=4)
    ax_xy.scatter([rx_per[i, -1]], [ry_per[i, -1]], s=30, marker="s",
                  facecolor=colors[i % len(colors)], edgecolor="k",
                  lw=0.5, zorder=4)
# Intensity-sum (combined) trajectory
sc = ax_xy.scatter(rx, ry, c=hinge_deg_arr, cmap="viridis", s=14, zorder=3)
ax_xy.plot(rx, ry, "-", color="0.3", lw=1.2, alpha=0.9, label="sum")
R_SAT_local = _C / (_A + _B)
tt = np.linspace(0, 2 * np.pi, 360)
ax_xy.plot(R_SAT_local * np.cos(tt), R_SAT_local * np.sin(tt),
           ":", color="0.6", lw=1, label=f"|r|=R_SAT={R_SAT_local:.3f}")
ax_xy.scatter([rx[0]],  [ry[0]],  s=70, marker="o",
              facecolor="white", edgecolor="k", zorder=5,
              label=f"sum hinge={hinge_deg_arr[0]:.0f}°")
ax_xy.scatter([rx[-1]], [ry[-1]], s=70, marker="s",
              facecolor="white", edgecolor="k", zorder=5,
              label=f"sum hinge={hinge_deg_arr[-1]:.0f}°")
ax_xy.set_aspect("equal")
ax_xy.set_xlabel(r"$r_x$"); ax_xy.set_ylabel(r"$r_y$")
ax_xy.set_title(r"$(r_x, r_y)$ trajectories: per rod + sum")
ax_xy.grid(alpha=0.25); ax_xy.legend(fontsize=6, loc="upper right")
plt.colorbar(sc, ax=ax_xy, label="hinge (deg)", shrink=0.85)

# (4) θ recovery
ax_t = fig.add_subplot(2, 3, 4)
for i in range(n_rods):
    ax_t.plot(hinge_deg_arr, np.degrees(theta_lab_per[i]),
              color=colors[i % len(colors)], lw=1.2, alpha=0.85,
              label=f"rod {i} truth")
ax_t.plot(hinge_deg_arr, np.degrees(theta_rec),
          color="k", lw=2, ls="--", label="single-rod recovery")
ax_t.set_xlabel("hinge angle (deg)")
ax_t.set_ylabel(r"$\theta_{\rm lab}$ (deg)")
ax_t.set_title(r"$\theta_{\rm lab}$: per-rod truth vs recovery")
ax_t.grid(alpha=0.25); ax_t.legend(fontsize=7)

# (5) φ recovery (wrap into [-90°, 90°] since inverse only resolves φ mod π)
def _wrap_phi_pi(phi_rad):
    return ((phi_rad + np.pi / 2) % np.pi) - np.pi / 2

ax_p = fig.add_subplot(2, 3, 5)
for i in range(n_rods):
    ax_p.plot(hinge_deg_arr, np.degrees(_wrap_phi_pi(phi_lab_per[i])),
              color=colors[i % len(colors)], lw=1.2, alpha=0.85,
              label=f"rod {i} truth")
ax_p.plot(hinge_deg_arr, np.degrees(phi_rec),
          color="k", lw=2, ls="--", label="single-rod recovery")
ax_p.set_xlabel("hinge angle (deg)")
ax_p.set_ylabel(r"$\phi_{\rm lab}\ {\rm mod}\ \pi$ (deg)")
ax_p.set_title(r"$\phi_{\rm lab}$: per-rod truth vs recovery")
ax_p.grid(alpha=0.25); ax_p.legend(fontsize=7)

# (6) recovered |r|
ax_r = fig.add_subplot(2, 3, 6)
ax_r.plot(hinge_deg_arr, r_amp, color="tab:red", lw=2, label=r"$|r|=\sqrt{r_x^2+r_y^2}$")
ax_r.axhline(R_SAT_local, color="0.5", ls="--", lw=1,
             label=f"R_SAT = {R_SAT_local:.3f}")
ax_r.set_xlabel("hinge angle (deg)")
ax_r.set_ylabel(r"$|r|$")
ax_r.set_title(r"Anisotropy magnitude $|r|$ vs hinge")
ax_r.grid(alpha=0.25); ax_r.legend(fontsize=8)

fig.suptitle(
    f"Manual rod ensemble (n={n_rods}, fixed in arm frame).  "
    "Sum 4-channel intensities, derive $(r_x, r_y)$, then invert as if a single rod.",
    fontsize=11,
)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()